# EP-FedProto v3 — NB3: Ablations, Multi-Seed Core Comparison, Non-IID & Device-Tier Sensitivity, Failure-Case Analysis

**Prerequisite:** NB2 output (`nb2_bundle`, from `ep_fedproto_nb2_v2`) added as input data.

**Scope per the final plan:**
1. Existing 9 ablations (unchanged from the prior PGFCL-lineage NB3, v13) — SSL,
   prototype, FedProx, ContribAgg, GradGuard, focal loss, label propagation,
   FedPer, EMA — plus a `Full-Backbone (CFG)` reference row.
2. 3 new EP-FedProto-specific ablation arms: `No-Nesting` (single-budget loss,
   rank-aware aggregator kept), `Uniform-Agg` (nested loss kept, plain
   size-weighted dimension-wise aggregation), `Full-EP-FedProto` (reference,
   aliased from the NB2 headline run).
3. Mandatory multi-seed core comparison: extends EP-FedProto and FjORD's
   headline runs from NB2's 3 seeds to the full 5-seed budget FedProto already
   has, then reports mean ± std, a paired t-test, a Wilcoxon signed-rank test,
   and Cohen's d for EP-FedProto vs FedProto and EP-FedProto vs FjORD.
4. Non-IID severity sweep: a new `temporal_federated_split_severity` split
   that interpolates between the original pure-chronological split (maximal
   non-IID) and an approximately-IID random split, swept across 4 severities.
5. Skewed device-tier distribution (60/30/10, i.e. 60% of clients at the
   smallest tier, 30% at the next, 10% at the third, 0% at the largest).
6. Failure-case analysis: a concrete, mathematically-demonstrated failure mode
   of `rank_aware_contrib_agg` in sparse high-dim segments (singleton-eligible
   segments can't be down-weighted, regardless of quality score), verified
   both synthetically and against the skewed-fleet run above.

**v3-NB3 fixes vs the old v13 (documented inline at each site, not hidden):**
- Removed `Diag A3` ('in-round SSL restored'): it referenced an
  `ExperimentConfig(use_ssl_inround=True)` flag that no longer exists on
  `ExperimentConfig` (in-round SSL was removed entirely upstream) — this would
  raise `TypeError` if run as-is.
- Renamed `'Full PGFCL v10'` → `'Full-Backbone (CFG)'` and switched it from an
  (invalid, in this project's bundle chain) alias to `pgfcl_runs` — which this
  lineage's `nb2_bundle` never contains — to a verified alias of `Diag A2`
  (provably the same config by hash), avoiding both a crash and a third
  redundant training run of the same config.

**After finish:** Save & Run All → add this notebook's output as input to NB4
(headline plots, compute/communication tables, Pareto figure, scalability
plot, non-IID + failure-case figures, edge-device results, packaging).

In [ ]:
# ── Cell 1: Pinned PyG install ─────────────────────────────────────────────
import subprocess, sys, importlib

def get_torch_cuda_tag():
    import torch
    tv = torch.__version__.split('+')[0]
    cv = torch.version.cuda
    if cv is None:
        return tv, 'cpu'
    major, minor = cv.split('.')[:2]
    return tv, f'cu{major}{minor}'

torch_ver, cuda_tag = get_torch_cuda_tag()
print(f'PyTorch {torch_ver} | CUDA tag: {cuda_tag}')

WHL_URL  = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
PACKAGES = ['torch-geometric','torch-scatter','torch-sparse',
            'torch-cluster','torch-spline-conv']

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       *PACKAGES, '-f', WHL_URL])

missing = []
for pkg in ['torch_geometric','torch_scatter','torch_sparse',
            'torch_cluster','torch_spline_conv']:
    try:
        importlib.import_module(pkg)
        print(f'  OK  {pkg}')
    except ImportError:
        missing.append(pkg)
if missing:
    raise RuntimeError(f'Failed to install: {missing}')
print('All PyG packages installed.')


In [ ]:
import os, glob, random, copy, time, warnings, hashlib
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Tuple, Dict
from scipy import stats as scipy_stats

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch_geometric.data import Data
# v9: GATv2Conv replaces GATConv (query-dependent attention, Brody et al. 2022)
from torch_geometric.nn import GATv2Conv, SAGEConv
from sklearn.metrics import (f1_score, roc_auc_score, accuracy_score,
                              precision_score, recall_score, confusion_matrix,
                              precision_recall_curve)
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


@dataclass
class ExperimentConfig:
    # Architecture
    hidden:       int   = 128
    emb_dim:      int   = 64
    heads:        int   = 4
    dropout:      float = 0.4
    head_dropout: float = 0.3

    # Federation
    n_clients:  int   = 4
    seeds:      tuple = (42, 123, 7, 456, 789)  # FIX A2: 5 seeds for adequate statistical power
    test_ratio: float = 0.3

    # FIX A1: feature dimensionality flag
    use_dev_features: bool = True  # True=330-dim (raw+dev), False=165-dim (raw only)

    # Training schedule
    global_rounds:        int   = 100
    head_finetune_rounds: int   = 20
    ssl_pretrain_rounds:  int   = 6    # pre-warm only — NOT run per round
    ssl_epochs:           int   = 20
    sup_epochs:           int   = 25
    lr:                   float = 0.005
    lr_min:               float = 0.0005

    # FedProx
    # FIX: mu_encoder lowered 0.05→0.01 (sweep showed 0.05 is on steep downslope)
    mu_encoder: float = 0.01
    mu_head:    float = 0.0

    # SSL — used ONLY during the pre-warm phase, never in the main training loop
    # FIX: tau raised 0.4→0.7 (sweep best); lam_warmup_rounds lowered 5→2 (sweep best)
    tau:            float = 0.7
    ssl_edge_floor: int   = 15
    feat_drop:      float = 0.3
    edge_drop:      float = 0.2

    # Prototype aggregation
    lam_max:              float = 0.6
    lam_warmup_rounds:    int   = 2
    ema_momentum:         float = 0.85
    raw_blend:            float = 0.2
    use_degree_weighting: bool  = True

    # Ablation flags
    use_ssl:     bool = True   # controls pre-warm phase; in-round SSL removed entirely
    use_protos:  bool = True
    use_fedprox: bool = True

    # v8 contribution flags
    use_contrib_agg:  bool  = True
    contrib_floor:    float = 0.1
    use_grad_guard:   bool  = True
    anomaly_thresh:   float = -0.1
    use_saliency:     bool  = True
    saliency_top_k:   int   = 20
    use_calibration:  bool  = True
    ece_bins:         int   = 15

    # v9 flags
    # FIX: focal_gamma lowered 2.0→1.0 (sweep monotonically decreasing; 1.0 best)
    use_focal_loss:   bool  = True
    focal_gamma:      float = 1.0

    # FIX: use_label_prop disabled (No-LabelProp ablation scores higher than full model)
    # lp_alpha raised to 0.9 (best in sweep) so it is safe to re-enable later.
    use_label_prop:   bool  = False
    lp_alpha:         float = 0.9
    lp_steps:         int   = 1      # also reduced from 2 → 1 for when LP is re-enabled

    # FedPer: local-only head fine-tuning, NO head averaging
    use_fedper:       bool  = True

    cosine_T0:        int   = 50

    # Baselines
    baseline_rounds:       int = 50
    baseline_local_epochs: int = 25
    centralized_epochs:    int = 150


def config_hash(cfg: ExperimentConfig) -> str:
    """Short deterministic hash of config — used to detect checkpoint key collisions."""
    d = {k: v for k, v in asdict(cfg).items() if k != 'seeds'}
    raw = str(sorted(d.items())).encode()
    return hashlib.md5(raw).hexdigest()[:8]


CFG = ExperimentConfig()
print(f'Device : {DEVICE}')
print(f'Seeds  : {CFG.seeds}')
print(f'Config hash: {config_hash(CFG)}')
print(f'v10 fixes: mu_encoder={CFG.mu_encoder}, tau={CFG.tau}, ')
print(f'           lam_warmup={CFG.lam_warmup_rounds}, focal_gamma={CFG.focal_gamma}, ')
print(f'           use_label_prop={CFG.use_label_prop}, lp_alpha={CFG.lp_alpha}')
print(f'           in-round SSL: REMOVED (pre-warm only)')


In [ ]:
import pickle, os, glob

LOCAL_CKPT_DIR = '/kaggle/working/pgfcl_checkpoints'
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

INPUT_CKPT_DIRS = []
if os.path.exists('/kaggle/input'):
    for hit in glob.glob('/kaggle/input/**/pgfcl_checkpoints', recursive=True):
        if os.path.isdir(hit) and hit not in INPUT_CKPT_DIRS:
            INPUT_CKPT_DIRS.append(hit)
            print(f'  [ckpt] found: {hit}  ({len(os.listdir(hit))} files)')

if not INPUT_CKPT_DIRS:
    print('  [ckpt] No prior checkpoint dirs found in /kaggle/input.')


def _fname(key):
    return key.replace(' ', '_').replace('/', '_') + '.pkl'


def _find(key):
    p = os.path.join(LOCAL_CKPT_DIR, _fname(key))
    if os.path.exists(p):
        return p
    for d in INPUT_CKPT_DIRS:
        p = os.path.join(d, _fname(key))
        if os.path.exists(p):
            return p
    return None


def save_ckpt(key, obj):
    path = os.path.join(LOCAL_CKPT_DIR, _fname(key))
    with open(path, 'wb') as fh:
        pickle.dump(obj, fh, protocol=4)
    kb = os.path.getsize(path) / 1024
    print(f'  [ckpt] saved  -> {key}  ({kb:.0f} KB)')


def load_ckpt(key):
    p = _find(key)
    if p is not None:
        with open(p, 'rb') as fh:
            obj = pickle.load(fh)
        src = 'local' if LOCAL_CKPT_DIR in p else 'prior-run'
        print(f'  [ckpt] loaded <- {key}  ({src})')
        return obj
    return None


def run_or_load(key, fn, expected_hash=None):
    """Load checkpoint if available, validating config hash to prevent stale-config collisions.

    Pass expected_hash=config_hash(CFG) to reject checkpoints saved under a
    different config (e.g. a prior version with different hyperparameters).
    """
    obj = load_ckpt(key)
    if obj is not None:
        if expected_hash is not None:
            stored_hash = obj.get('config_hash') if isinstance(obj, dict) else None
            if stored_hash is not None and stored_hash != expected_hash:
                print(f'  [ckpt] WARNING: hash mismatch for {key!r} '
                      f'(stored={stored_hash}, expected={expected_hash}) — recomputing')
                obj = None
    if obj is not None:
        return obj
    obj = fn()
    save_ckpt(key, obj)
    return obj


def list_ckpts():
    local = sorted(os.listdir(LOCAL_CKPT_DIR))
    print(f'  Local  ({LOCAL_CKPT_DIR}): {len(local)} file(s)')
    for f in local:
        kb = os.path.getsize(os.path.join(LOCAL_CKPT_DIR, f)) / 1024
        print(f'    {f}  ({kb:.0f} KB)')
    for d in INPUT_CKPT_DIRS:
        files = sorted(os.listdir(d))
        print(f'  Input  ({d}): {len(files)} file(s)')
        for f in files:
            kb = os.path.getsize(os.path.join(d, f)) / 1024
            print(f'    {f}  ({kb:.0f} KB)')


list_ckpts()
print('Checkpoint system ready.')


## Dataset

In [ ]:
ELLIPTIC_DIR = os.environ.get(
    'ELLIPTIC_DIR',
    '/kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset'
)
if not os.path.isfile(os.path.join(ELLIPTIC_DIR, 'elliptic_txs_features.csv')):
    hits = glob.glob('/kaggle/input/**/elliptic_txs_features.csv', recursive=True)
    if hits:
        ELLIPTIC_DIR = os.path.dirname(hits[0])
    else:
        for root, _, files in os.walk('/kaggle/input'):
            if 'elliptic_txs_features.csv' in files:
                ELLIPTIC_DIR = root; break
        else:
            raise FileNotFoundError('Cannot find elliptic_txs_features.csv')
print(f'Elliptic dir: {ELLIPTIC_DIR}')


def load_elliptic(data_dir: str = None, use_dev_features: bool = True) -> Data:
    if data_dir is None:
        data_dir = ELLIPTIC_DIR
    print('Loading Elliptic dataset...')
    features = pd.read_csv(f'{data_dir}/elliptic_txs_features.csv', header=None)
    edges    = pd.read_csv(f'{data_dir}/elliptic_txs_edgelist.csv')
    classes  = pd.read_csv(f'{data_dir}/elliptic_txs_classes.csv')

    node_ids  = features.iloc[:, 0].values
    id2idx    = {nid: i for i, nid in enumerate(node_ids)}
    timesteps = features.iloc[:, 1].values.astype(int)
    raw_feats = features.iloc[:, 2:].values.astype(np.float32)

    # FIX: some nodes in the Elliptic CSV have NaN feature values.
    # A single NaN propagates through the per-timestep z-score into the entire
    # timestep block (20 nodes) and then through StandardScaler → inf/NaN inputs
    # to the GNN, causing the "input has nan" error.
    # Replace NaN/inf in raw features with 0 before any further processing.
    raw_feats = np.nan_to_num(raw_feats, nan=0.0, posinf=0.0, neginf=0.0)

    # Per-timestep z-score deviation features (165→330 dims)
    # NOTE for paper: all methods receive 330-dim input; not directly comparable
    # to published results on raw 165-dim features.
    dev_feats = np.zeros_like(raw_feats)
    for t in np.unique(timesteps):
        mask = timesteps == t
        mu   = raw_feats[mask].mean(0)
        sd   = raw_feats[mask].std(0) + 1e-8
        dev_feats[mask] = (raw_feats[mask] - mu) / sd
    # FIX A1: conditional feature concatenation based on use_dev_features flag
    if use_dev_features:
        all_feats = np.concatenate([raw_feats, dev_feats], axis=1)  # 330-dim
    else:
        all_feats = raw_feats.copy()  # 165-dim (for published baseline comparison)
    # Safety clamp after StandardScaler in case any column is still degenerate
    sc = StandardScaler()
    all_feats = sc.fit_transform(all_feats).astype(np.float32)
    all_feats = np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)

    classes['class'] = classes['class'].map({'1': 1, '2': 0, 'unknown': -1})
    label_map = dict(zip(classes['txId'], classes['class']))
    labels    = np.array([label_map.get(nid, -1) for nid in node_ids])

    valid_edges = [
        (id2idx[u], id2idx[v])
        for u, v in zip(edges.iloc[:, 0], edges.iloc[:, 1])
        if u in id2idx and v in id2idx
    ]
    n_dropped = len(edges) - len(valid_edges)
    if n_dropped:
        print(f'  Warning: dropped {n_dropped} edges')
    srcs, dsts = zip(*valid_edges) if valid_edges else ([], [])
    edge_index = torch.tensor([list(srcs), list(dsts)], dtype=torch.long)

    data = Data(
        x         = torch.tensor(all_feats),
        edge_index = edge_index,
        y          = torch.tensor(labels, dtype=torch.long),
        timestep   = torch.tensor(timesteps, dtype=torch.long)
    )
    print(f'  Nodes: {data.num_nodes:,} | Edges: {data.num_edges:,} | '
          f'Features: {data.num_node_features}')
    print(f'  Illicit: {(labels==1).sum():,} | '
          f'Licit: {(labels==0).sum():,} | '
          f'Unknown: {(labels==-1).sum():,}')
    return data


elliptic_data = load_elliptic(use_dev_features=True)
# FIX A1: also load 165-dim version for published-baseline comparison (C2)
elliptic_data_165 = load_elliptic(use_dev_features=False)
print(f'330-dim data: {elliptic_data.num_node_features} features')
print(f'165-dim data: {elliptic_data_165.num_node_features} features')


## Temporal Federated Split

In [ ]:
# CAVEAT: temporal split produces non-IID clients (different time windows).
# Clients 0 and 3 have ~350-470 illicit examples; clients 1 and 2 have ~1150.
# This is a realistic AML setting but gives the federated model a genuine
# structural disadvantage vs centralised (which sees all time windows together).

def make_mask(n: int, idx: np.ndarray) -> torch.Tensor:
    m = torch.zeros(n, dtype=torch.bool)
    if len(idx):
        m[torch.tensor(np.array(idx), dtype=torch.long)] = True
    return m


def temporal_federated_split(data: Data, cfg: ExperimentConfig):
    labels    = data.y.numpy()
    timesteps = data.timestep.numpy()
    valid_idx  = np.where(labels >= 0)[0]
    sorted_idx = valid_idx[np.argsort(timesteps[valid_idx])]
    splits     = np.array_split(sorted_idx, cfg.n_clients)

    def strat_split(arr):
        if len(arr) == 0:
            return arr, arr
        n_te = max(1, int(len(arr) * cfg.test_ratio))
        return arr[:-n_te], arr[-n_te:]

    clients = []
    for i, split in enumerate(splits):
        illicit        = split[labels[split] == 1]
        licit          = split[labels[split] == 0]
        ill_tr, ill_te = strat_split(illicit)
        lic_tr, lic_te = strat_split(licit)
        tr = np.concatenate([ill_tr, lic_tr])
        te = np.concatenate([ill_te, lic_te])
        clients.append({
            'id':         i,
            'train_mask': make_mask(data.num_nodes, tr),
            'test_mask':  make_mask(data.num_nodes, te),
            'n_train':    len(tr),
            'n_test':     len(te),
        })
        print(f'  Client {i}: train={len(tr):5d} test={len(te):4d} '
              f'ill_train={len(ill_tr):4d} ill_test={len(ill_te):3d}')
    return clients


def get_local_edge_index(data: Data, mask: torch.Tensor,
                          device: torch.device) -> torch.Tensor:
    """Edges where BOTH endpoints are inside mask."""
    ei   = data.edge_index.to(device)
    m    = mask.to(device)
    keep = m[ei[0]] & m[ei[1]]
    return ei[:, keep]


def get_inductive_edge_index(data, train_mask, test_mask, device):
    """Inductive test edges — at least one endpoint in test_mask."""
    ei    = data.edge_index.to(device)
    tr    = train_mask.to(device)
    te    = test_mask.to(device)
    all_m = tr | te
    keep  = (te[ei[1]] & all_m[ei[0]]) | (te[ei[0]] & te[ei[1]])
    return ei[:, keep]


print('=== Elliptic — Temporal Train/Test Split ===')
elliptic_clients = temporal_federated_split(elliptic_data, CFG)


## Models

In [ ]:
class SAGEGATEncoder(nn.Module):
    """
    Encoder: SAGEConv ×2 (skip connections) → GATv2Conv (query-dependent attention).
    forward_with_attention() returns GAT attention weights for saliency logging
    at zero extra compute cost.
    """

    def __init__(self, in_dim: int, cfg: ExperimentConfig):
        super().__init__()
        h, e, dr = cfg.hidden, cfg.emb_dim, cfg.dropout
        self.dropout = dr
        self.conv1 = SAGEConv(in_dim, h)
        self.conv2 = SAGEConv(h, h)
        self.conv3 = GATv2Conv(h, e, heads=1, dropout=dr, concat=False)
        self.skip1 = nn.Linear(in_dim, h, bias=False)
        self.skip2 = nn.Linear(h, h, bias=False)
        self.bn1   = nn.BatchNorm1d(h)
        self.bn2   = nn.BatchNorm1d(h)
        self.proj_head = nn.Sequential(
            nn.Linear(e, e * 2), nn.ReLU(), nn.Linear(e * 2, e)
        )
        self.raw_projector = nn.Linear(in_dim, e, bias=False)

    def _sage_layers(self, x, edge_index):
        h1 = self.conv1(x, edge_index) + self.skip1(x)
        h1 = F.relu(self.bn1(h1))
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.conv2(h1, edge_index) + self.skip2(h1)
        h2 = F.relu(self.bn2(h2))
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        return h2

    def forward(self, x, edge_index):
        h2 = self._sage_layers(x, edge_index)
        return self.conv3(h2, edge_index)

    def forward_with_attention(self, x, edge_index):
        h2 = self._sage_layers(x, edge_index)
        emb, (att_edge_index, att_weights) = self.conv3(
            h2, edge_index, return_attention_weights=True
        )
        return emb, att_edge_index, att_weights

    def encode_with_proj(self, x, edge_index):
        z = self.forward(x, edge_index)
        return z, self.proj_head(z)

    def forward_od(self, x, edge_index, width_ratio: float):
        """Ordered Dropout (FjORD) forward pass. Zeroes the tail channels of
        every hidden/output layer beyond width_ratio * layer_width, in a fixed
        channel order, so a low-budget client's active sub-network is always a
        strict nested prefix of every larger client's sub-network. Masked-out
        output channels get exactly zero gradient through the weight rows that
        produce them (verified in isolation), so plain size-weighted FedAvg on
        the full state_dict is a correct aggregator here -- an untouched
        weight row is simply left at the value the client started the round
        with, not corrupted noise.
        """
        def _od_mask(h):
            keep = max(1, round(h.shape[1] * width_ratio))
            mask = torch.zeros_like(h)
            mask[:, :keep] = 1.0
            return h * mask

        h1 = self.conv1(x, edge_index) + self.skip1(x)
        h1 = F.relu(self.bn1(h1))
        h1 = _od_mask(h1)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.conv2(h1, edge_index) + self.skip2(h1)
        h2 = F.relu(self.bn2(h2))
        h2 = _od_mask(h2)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        emb = self.conv3(h2, edge_index)
        emb = _od_mask(emb)
        return emb


class ClassHead(nn.Module):
    def __init__(self, in_dim: int, cfg: ExperimentConfig, n_classes: int = 2):
        super().__init__()
        h1 = max(128, in_dim * 2)
        h2 = h1 // 2
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(cfg.head_dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(cfg.head_dropout / 2),
            nn.Linear(h2, n_classes),
        )

    def forward(self, x):
        return self.net(x)


class FullGAT(nn.Module):
    def __init__(self, in_dim: int, cfg: ExperimentConfig):
        super().__init__()
        self.encoder = SAGEGATEncoder(in_dim, cfg)
        self.head    = ClassHead(cfg.emb_dim, cfg)

    def forward(self, x, edge_index, trunc_dim: int = None):
        """trunc_dim: EP-FedProto v3 fixed-tier baseline (BUGFIX). When set,
        zeroes embedding dims >= trunc_dim BEFORE the classification head --
        same post-activation masking principle as encoder.forward_od (FjORD)
        -- so the tier constraint actually restricts what the head can see,
        instead of only touching the auxiliary prototype loss like before.
        None = full emb_dim, unchanged v10 behaviour."""
        emb = self.encoder(x, edge_index)
        if trunc_dim is not None:
            mask = torch.zeros_like(emb)
            mask[:, :trunc_dim] = 1.0
            emb = emb * mask
        return self.head(emb), emb


class SAGEModel(nn.Module):
    """SAGE baseline — same dims as PGFCL for fair comparison."""

    def __init__(self, in_dim: int, cfg: ExperimentConfig, n_classes: int = 2):
        super().__init__()
        h, e = cfg.hidden, cfg.emb_dim
        self.conv1   = SAGEConv(in_dim, h)
        self.conv2   = SAGEConv(h, h)
        self.conv3   = SAGEConv(h, e)
        self.skip1   = nn.Linear(in_dim, h, bias=False)
        self.skip2   = nn.Linear(h, h, bias=False)
        self.bn1     = nn.BatchNorm1d(h)
        self.bn2     = nn.BatchNorm1d(h)
        self.head    = nn.Sequential(
            nn.Linear(e, e * 2), nn.ReLU(), nn.Linear(e * 2, n_classes)
        )
        self.dropout = cfg.dropout

    def forward(self, x, edge_index):
        h1 = F.relu(self.bn1(self.conv1(x, edge_index) + self.skip1(x)))
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = F.relu(self.bn2(self.conv2(h1, edge_index) + self.skip2(h1)))
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        emb = self.conv3(h2, edge_index)
        return self.head(emb), emb

GATEncoder = SAGEGATEncoder
print('Models defined.')


## Utilities

In [ ]:
# ── Evaluation ───────────────────────────────────────────────────────────────

def evaluate_tuned(model, data, train_mask, test_mask, device, cfg=None,
                    proto_dim: int = None):
    """
    Threshold tuned on train set (no leakage). Test inference is inductive.
    Label propagation is applied only if cfg.use_label_prop is True.
    FIX: LP disabled by default (use_label_prop=False) — ablation showed it
    hurts at any reasonable alpha on this graph topology.

    proto_dim: EP-FedProto v3 fixed-tier baseline (BUGFIX). Forwarded as
    trunc_dim into the model so evaluation actually sees the same truncated
    embedding the tier is supposed to constrain, instead of always scoring
    the full-capacity model regardless of tier_d. None = unchanged v10
    behaviour.
    """
    if cfg is None:
        cfg = CFG
    model.eval()

    ei_tr = get_local_edge_index(data, train_mask, device)
    with torch.no_grad():
        logits_tr, _ = model(data.x.to(device), ei_tr, trunc_dim=proto_dim)
    probs_tr = F.softmax(logits_tr, dim=1)[:, 1].cpu().numpy()
    # Guard: NaN in probs means the model has diverged (BN with tiny batch, etc.)
    if not np.isfinite(probs_tr).all():
        return {'acc': 0., 'f1': 0., 'auc': 0., 'prec': 0., 'rec': 0.,
                'cm': np.zeros((2, 2), int), 'thresh': 0.5}

    tr_lab = train_mask & (data.y >= 0)
    best_thresh = 0.5
    if tr_lab.sum() > 0 and len(np.unique(data.y[tr_lab].numpy())) > 1:
        p, r, thresholds = precision_recall_curve(
            data.y[tr_lab].numpy(), probs_tr[tr_lab.numpy()])
        f1s = 2 * p * r / (p + r + 1e-8)
        best_thresh = float(np.clip(thresholds[np.argmax(f1s[:-1])], 0.1, 0.9))

    ei_te = get_inductive_edge_index(data, train_mask, test_mask, device)
    with torch.no_grad():
        logits_te, _ = model(data.x.to(device), ei_te, trunc_dim=proto_dim)
    probs_te_full = F.softmax(logits_te, dim=1)[:, 1]

    if cfg.use_label_prop:
        probs_te_full = label_propagation(
            probs_te_full, ei_te, data.num_nodes,
            alpha=cfg.lp_alpha, steps=cfg.lp_steps
        ).to(device)
    probs_te = probs_te_full.cpu().numpy()

    te_lab = test_mask & (data.y >= 0)
    if te_lab.sum() == 0:
        return {'acc': 0., 'f1': 0., 'auc': 0., 'prec': 0., 'rec': 0.,
                'cm': np.zeros((2, 2), int), 'thresh': best_thresh}

    probs_te_masked = probs_te[te_lab.numpy()]
    preds_te        = (probs_te_masked >= best_thresh).astype(int)
    true_te         = data.y[te_lab].numpy()

    return {
        'acc':    accuracy_score(true_te, preds_te),
        'f1':     f1_score(true_te, preds_te, zero_division=0),
        'auc':    roc_auc_score(true_te, probs_te_masked) if len(np.unique(true_te)) > 1 else 0.,
        'prec':   precision_score(true_te, preds_te, zero_division=0),
        'rec':    recall_score(true_te, preds_te, zero_division=0),
        'cm':     confusion_matrix(true_te, preds_te),
        'thresh': best_thresh
    }


# ── Loss Functions ────────────────────────────────────────────────────────────

def weighted_ce(logits, labels, device):
    n_classes = logits.shape[1]
    counts    = torch.bincount(labels, minlength=n_classes).float().clamp(min=1.0)
    weight    = labels.shape[0] / (n_classes * counts)
    weight    = weight.clamp(0.1, 10.0)
    return F.cross_entropy(logits, labels, weight=weight.to(device))


def focal_loss(logits, labels, device, gamma=1.0):
    """
    Focal loss (Lin et al. 2017) with inverse-frequency class weights.
    FIX: default gamma lowered 2.0→1.0. Sweep showed monotonic decrease
    in F1 as gamma increases; gamma=1.0 is best. gamma=2.0 over-suppresses
    easy examples on this class-imbalanced, non-IID federated setting.
    """
    n_classes = logits.shape[1]
    counts    = torch.bincount(labels, minlength=n_classes).float().clamp(1.0)
    alpha_cls = (labels.shape[0] / (n_classes * counts)).clamp(0.1, 10.0)
    alpha_t   = alpha_cls[labels]
    ce  = F.cross_entropy(logits, labels, reduction='none')
    pt  = torch.exp(-ce).clamp(1e-7, 1.0 - 1e-7)  # clamp prevents (1-pt)^gamma=0 → NaN grad
    fl  = alpha_t * (1.0 - pt) ** gamma * ce
    return fl.mean()


def supervised_loss(logits, labels, device, cfg):
    if cfg.use_focal_loss:
        return focal_loss(logits, labels, device, gamma=cfg.focal_gamma)
    return weighted_ce(logits, labels, device)


def label_propagation(probs, edge_index, n_nodes, alpha=0.9, steps=1):
    """
    Post-processing LP. Disabled by default (use_label_prop=False).
    FIX: alpha default raised 0.8→0.9, steps lowered 2→1 for when it
    is re-enabled. The sweep showed LP becomes harmful below alpha~0.9
    on the inductive test subgraph due to unlabeled-region noise.
    """
    dev = probs.device
    ei  = edge_index.to(dev)
    src_idx, dst_idx = ei[0], ei[1]
    p   = probs.clone()
    p0  = probs.clone()
    deg = torch.zeros(n_nodes, device=dev)
    deg.scatter_add_(0, dst_idx, torch.ones(dst_idx.shape[0], device=dev))
    deg = deg.clamp(min=1.0)
    for _ in range(steps):
        agg = torch.zeros(n_nodes, device=dev)
        agg.scatter_add_(0, dst_idx, p[src_idx])
        agg = agg / deg
        p = alpha * p0 + (1.0 - alpha) * agg
    return p


def infonce_loss(z_proj, edge_index, local_mask, device, cfg: ExperimentConfig):
    z_norm    = F.normalize(z_proj, dim=1)
    feat_mask = torch.rand(z_proj.shape[1], device=device) > cfg.feat_drop
    z_aug     = F.normalize(z_norm * feat_mask.float(), dim=1)
    src, dst = edge_index
    keep     = torch.rand(src.shape[0], device=device) > cfg.edge_drop
    src_k, dst_k = src[keep], dst[keep]
    if src_k.numel() == 0:
        return torch.tensor(0.0, device=device)
    pos_sim = (z_norm[src_k] * z_aug[dst_k]).sum(1) / cfg.tau
    anchor_nodes = torch.cat([src_k, dst_k]).unique()
    local_nodes  = torch.where(local_mask.to(device))[0]
    neg_pool     = local_nodes[~torch.isin(local_nodes, anchor_nodes)]
    if neg_pool.numel() < 4:
        perm     = torch.randperm(src_k.shape[0], device=device)
        neg_pool = src_k[perm]
    n_neg   = min(256, neg_pool.shape[0])
    neg_idx = neg_pool[torch.randperm(neg_pool.shape[0], device=device)[:n_neg]]
    neg_sim = torch.mm(z_norm[src_k], z_norm[neg_idx].T) / cfg.tau
    logits = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1)
    target = torch.zeros(logits.shape[0], dtype=torch.long, device=device)
    return F.cross_entropy(logits, target)


def label_aware_edge_supcon(z_proj, data, local_mask, device, cfg):
    """Label-aware SupCon: illicit anchors, illicit positives, licit negatives."""
    labels = data.y.to(device)
    lm = local_mask.to(device)
    ill_mask = lm & (labels == 1)
    lic_mask = lm & (labels == 0)
    if ill_mask.sum() < 2 or lic_mask.sum() < 1:
        return torch.tensor(0.0, device=device)
    z_norm = F.normalize(z_proj, dim=1)
    feat_mask = torch.rand(z_proj.shape[1], device=device) > cfg.feat_drop
    z_aug = F.normalize(z_norm * feat_mask.float(), dim=1)
    ill_idx = torch.where(ill_mask)[0]
    lic_idx = torch.where(lic_mask)[0]
    losses = []
    n_anchors = min(64, ill_idx.shape[0])
    perm = torch.randperm(ill_idx.shape[0], device=device)[:n_anchors]
    anchors = ill_idx[perm]
    for a in anchors:
        pos_pool = ill_idx[ill_idx != a]
        if pos_pool.shape[0] == 0:
            continue
        n_pos = min(8, pos_pool.shape[0])
        pos = pos_pool[torch.randperm(pos_pool.shape[0], device=device)[:n_pos]]
        n_neg = min(32, lic_idx.shape[0])
        neg = lic_idx[torch.randperm(lic_idx.shape[0], device=device)[:n_neg]]
        pos_sim = (z_norm[a] * z_aug[pos]).sum(1) / cfg.tau
        neg_sim = (z_norm[a] * z_aug[neg]).sum(1) / cfg.tau
        logits = torch.cat([pos_sim, neg_sim])
        target = torch.zeros(logits.shape[0], device=device)
        target[:pos_sim.shape[0]] = 1.0 / pos_sim.shape[0]
        log_probs = F.log_softmax(logits, dim=0)
        loss = -(target * log_probs).sum()
        losses.append(loss)
    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=device)


def prototype_supcon_loss(z, labels, mask, global_protos, device,
                           cfg: ExperimentConfig) -> torch.Tensor:
    labeled = mask.to(device) & (labels.to(device) >= 0)
    if not labeled.any() or global_protos is None:
        return torch.tensor(0.0, device=device)
    z_lab = F.normalize(z[labeled], dim=1)
    y_lab = labels.to(device)[labeled]
    p0 = F.normalize(global_protos[0].detach().unsqueeze(0), dim=1)
    p1 = F.normalize(global_protos[1].detach().unsqueeze(0), dim=1)
    protos_cat   = torch.cat([p0, p1], dim=0)
    proto_logits = torch.mm(z_lab, protos_cat.T) / cfg.tau
    return F.cross_entropy(proto_logits, y_lab)


# ── Aggregation Helpers ────────────────────────────────────────────────────────

def avg_metrics(metric_list):
    keys = ['f1', 'auc', 'prec', 'rec', 'acc']
    out  = {}
    for k in keys:
        vals = [m[k] for m in metric_list if k in m]
        out[k]           = float(np.mean(vals))
        out[k + '_std']  = float(np.std(vals))
        out[k + '_vals'] = vals
    return out


def fedavg_state_selective(global_model, local_models, client_sizes,
                            exclude_keys=None):
    if exclude_keys is None:
        exclude_keys = ['running_mean', 'running_var', 'num_batches_tracked']
    total   = sum(client_sizes)
    g_state = global_model.state_dict()
    new_state = {}
    for key in g_state:
        if any(ex in key for ex in exclude_keys):
            new_state[key] = g_state[key]
            continue
        acc = torch.zeros_like(g_state[key].float())
        for i, lm in enumerate(local_models):
            acc += lm.state_dict()[key].float() * (client_sizes[i] / total)
        new_state[key] = acc.to(g_state[key].dtype)
    global_model.load_state_dict(new_state)
    return global_model


def fedavg_encoder_only(global_model, local_encoders, client_sizes):
    exclude = ['running_mean', 'running_var', 'num_batches_tracked', 'proj_head']
    total   = sum(client_sizes)
    enc_ref = global_model.encoder.state_dict()
    new_enc = {}
    for key in enc_ref:
        if any(ex in key for ex in exclude):
            new_enc[key] = enc_ref[key]
            continue
        acc = torch.zeros_like(enc_ref[key].float())
        for i, enc in enumerate(local_encoders):
            acc += enc.state_dict()[key].float() * (client_sizes[i] / total)
        new_enc[key] = acc.to(enc_ref[key].dtype)
    global_model.encoder.load_state_dict(new_enc)
    return global_model


def compute_client_contrib_weights(client_emb_protos, client_sizes,
                                    global_protos, cfg):
    if global_protos is None:
        total = sum(client_sizes)
        return [s / total for s in client_sizes]
    global_ill = F.normalize(global_protos[1].unsqueeze(0), dim=1).squeeze(0)
    weights = []
    for protos, sz in zip(client_emb_protos, client_sizes):
        local_ill = F.normalize(protos[1].unsqueeze(0), dim=1).squeeze(0)
        cos_sim   = torch.dot(local_ill.float(), global_ill.float()).item()
        quality   = (cos_sim + 1.0) / 2.0
        quality   = cfg.contrib_floor + (1.0 - cfg.contrib_floor) * quality
        weights.append((sz ** 0.5) * quality)
    total = sum(weights)
    return [w / total for w in weights]


def fedavg_state_contrib(global_model, local_models, contrib_weights,
                          exclude_keys=None):
    if exclude_keys is None:
        exclude_keys = ['running_mean', 'running_var', 'num_batches_tracked']
    # Guard: if a client's model has NaN weights (diverged training),
    # exclude it and renormalise the remaining weights.
    nan_clients = []
    for i, lm in enumerate(local_models):
        params = [p for k, p in lm.state_dict().items() if not any(ex in k for ex in exclude_keys)]
        if any(torch.isnan(p.float()).any() for p in params):
            nan_clients.append(i)
    if nan_clients:
        print(f'  [fedavg] WARNING: clients {nan_clients} have NaN weights — excluded from aggregation')
        valid_w = [w if i not in nan_clients else 0.0 for i, w in enumerate(contrib_weights)]
        total_w = sum(valid_w)
        contrib_weights = [w / total_w if total_w > 0 else 1.0 / len(local_models) for w in valid_w]

    g_state   = global_model.state_dict()
    new_state = {}
    for key in g_state:
        if any(ex in key for ex in exclude_keys):
            new_state[key] = g_state[key]
            continue
        acc = torch.zeros_like(g_state[key].float())
        for i, lm in enumerate(local_models):
            if i in nan_clients:
                continue
            acc += lm.state_dict()[key].float() * contrib_weights[i]
        new_state[key] = acc.to(g_state[key].dtype)
    global_model.load_state_dict(new_state)
    return global_model


def compute_update_directions(global_model, local_models):
    exclude = ['running_mean', 'running_var', 'num_batches_tracked']
    g_state = global_model.state_dict()
    directions = []
    for lm in local_models:
        diff_parts = []
        for key in g_state:
            if any(ex in key for ex in exclude):
                continue
            diff = (lm.state_dict()[key].float() - g_state[key].float()).flatten()
            diff_parts.append(diff)
        vec = torch.cat(diff_parts)
        norm = vec.norm()
        directions.append(vec / (norm + 1e-8))
    return directions


def grad_guard_weights(global_model, local_models, client_sizes, cfg):
    directions  = compute_update_directions(global_model, local_models)
    total_size  = sum(client_sizes)
    size_w      = [s / total_size for s in client_sizes]
    consensus = sum(d * w for d, w in zip(directions, size_w))
    consensus = consensus / (consensus.norm() + 1e-8)
    weights, flagged = [], []
    for i, d in enumerate(directions):
        cos_sim = torch.dot(d, consensus).item()
        if cos_sim < cfg.anomaly_thresh:
            weights.append(cfg.contrib_floor)
            flagged.append(i)
        else:
            weights.append(size_w[i])
    total = sum(weights)
    weights = [w / total for w in weights]
    if flagged:
        print(f'  [GradGuard] flagged clients {flagged}')
    return weights, flagged


def extract_node_saliency(model, data, test_mask, device, cfg, top_k=None):
    if top_k is None:
        top_k = cfg.saliency_top_k
    model.eval()
    ei = get_local_edge_index(data, test_mask, device)
    x  = data.x.to(device)
    with torch.no_grad():
        emb, att_ei, att_w = model.encoder.forward_with_attention(x, ei)
        logits = model.head(emb)
        probs  = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
    att_w_flat = att_w.squeeze(-1).cpu()
    dst_nodes = att_ei[1].cpu()
    n_nodes   = data.num_nodes
    node_att  = torch.zeros(n_nodes)
    node_cnt  = torch.zeros(n_nodes)
    for j in range(dst_nodes.shape[0]):
        node_att[dst_nodes[j]] += att_w_flat[j].item()
        node_cnt[dst_nodes[j]] += 1
    node_saliency = node_att / (node_cnt + 1e-8)
    te_lab = test_mask & (data.y >= 0)
    te_idx = torch.where(te_lab)[0].cpu().numpy()
    saliency_te = node_saliency[te_idx].numpy()
    probs_te    = probs[te_idx]
    labels_te   = data.y[te_idx].numpy()
    risk_score  = probs_te * saliency_te
    top_idx     = np.argsort(risk_score)[::-1][:top_k]
    return {
        'node_indices': te_idx[top_idx],
        'risk_scores':  risk_score[top_idx],
        'probs':        probs_te[top_idx],
        'saliency':     saliency_te[top_idx],
        'true_labels':  labels_te[top_idx],
    }


def compute_ece(model, data, test_mask, device, cfg, train_mask=None,
                 proto_dim: int = None):
    """Compute Expected Calibration Error.

    Uses inductive edge index (same as evaluate_tuned) when train_mask is
    provided, ensuring ECE is measured under the same message-passing conditions
    as classification metrics. Falls back to transductive test subgraph if
    train_mask is not supplied.

    proto_dim: EP-FedProto v3 fixed-tier baseline (BUGFIX, same as
    evaluate_tuned) -- forwarded as trunc_dim so calibration is measured on
    the same truncated model as the F1 numbers, not the full-capacity one.
    """
    model.eval()
    if train_mask is not None:
        ei = get_inductive_edge_index(data, train_mask, test_mask, device)
    else:
        ei = get_local_edge_index(data, test_mask, device)
    with torch.no_grad():
        logits, _ = model(data.x.to(device), ei, trunc_dim=proto_dim)
    probs   = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
    te_lab  = test_mask & (data.y >= 0)
    y_true  = data.y[te_lab].numpy()
    y_prob  = probs[te_lab.numpy()]
    n_bins  = cfg.ece_bins
    bins    = np.linspace(0.0, 1.0, n_bins + 1)
    ece     = 0.0
    bin_data = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            bin_data.append({'conf': (lo+hi)/2, 'acc': 0., 'count': 0})
            continue
        conf = y_prob[mask].mean()
        acc  = y_true[mask].mean()
        frac = mask.sum() / len(y_true)
        ece += frac * abs(acc - conf)
        bin_data.append({'conf': conf, 'acc': acc, 'count': int(mask.sum())})
    return float(ece), bin_data


print('Utility functions loaded (v10 fixes: LP disabled, focal_gamma=1.0).')


## Training Functions

In [ ]:
# ── SSL pre-training (pre-warm phase ONLY) ────────────────────────────────────

def ssl_pretrain_client(encoder, data: Data, client: dict,
                         device: torch.device, cfg: ExperimentConfig,
                         opt: Optional[torch.optim.Optimizer] = None):
    """
    Label-aware SupCon on one client encoder.
    Full-graph message-passing; contrastive loss restricted to local labeled nodes.
    Called ONLY during ssl_pretrain_phase, never inside the main training loop.
    """
    encoder.train()
    if opt is None:
        opt = Adam(encoder.parameters(), lr=cfg.lr)
    x       = data.x.to(device)
    lm      = client['train_mask'].to(device)
    ei_full = data.edge_index.to(device)
    for _ in range(cfg.ssl_epochs):
        opt.zero_grad()
        _, z_proj = encoder.encode_with_proj(x, ei_full)
        loss = label_aware_edge_supcon(z_proj, data, lm, device, cfg)
        loss.backward()
        opt.step()


def ssl_pretrain_phase(global_model: FullGAT, data: Data, clients: list,
                        device: torch.device, cfg: ExperimentConfig,
                        verbose: bool = True) -> FullGAT:
    """
    FIX: SSL runs ONLY here (pre-warm), never inside the per-round loop.
    Diagnostic A showed in-round SSL (use_ssl=True during main loop) was
    net-negative: No-SSL variant scored F1=0.8125 vs full model F1=0.7655.
    The pre-warm phase alone (Diag A2) recovers all SSL benefit.
    """
    if not cfg.use_ssl or cfg.ssl_pretrain_rounds == 0:
        return global_model
    if verbose:
        print(f'  [SSL pre-warm] {cfg.ssl_pretrain_rounds} rounds (no in-round SSL after this)...')
    sizes = [c['n_train'] for c in clients]
    # Persist only the optimizer *state buffers* (step counts, moments) across
    # rounds, keyed by parameter position rather than tensor identity.
    # Storing full state_dicts over global_model.encoder params is wrong because
    # each round creates a fresh enc with new tensor identities — load_state_dict
    # would silently apply the wrong state. We instead carry raw state dicts and
    # transplant them by position after the new optimizer is created.
    client_ssl_opt_states: list = [None] * len(clients)
    for pre_rnd in range(cfg.ssl_pretrain_rounds):
        local_encoders = []
        for j, client in enumerate(clients):
            enc = SAGEGATEncoder(data.num_node_features, cfg).to(device)
            enc.load_state_dict(global_model.encoder.state_dict())
            opt = Adam(enc.parameters(), lr=cfg.lr)
            # Transplant saved state buffers by parameter position
            if client_ssl_opt_states[j] is not None:
                saved = client_ssl_opt_states[j]
                new_state = opt.state_dict()
                # Remap saved state from old param ids → new param ids
                old_id_to_pos = {pid: pos for pos, pid in enumerate(saved['state'])}
                for new_pid, pg in enumerate(new_state['param_groups'][0]['params']):
                    if new_pid in saved['state']:
                        new_state['state'][new_pid] = saved['state'][new_pid]
                try:
                    opt.load_state_dict(new_state)
                except (ValueError, KeyError):
                    pass  # mismatch on first round — start fresh, not a problem
            ssl_pretrain_client(enc, data, client, device, cfg, opt=opt)
            client_ssl_opt_states[j] = opt.state_dict()
            local_encoders.append(enc)
        global_model = fedavg_encoder_only(global_model, local_encoders, sizes)
        if verbose:
            print(f'    Pre-warm {pre_rnd+1}/{cfg.ssl_pretrain_rounds} done')
    return global_model


# ── Prototype computation ─────────────────────────────────────────────────────

def compute_embedding_prototypes(embeddings, data, mask, device, cfg):
    labels  = data.y.to(device)
    labeled = mask.to(device) & (labels >= 0)
    if cfg.use_degree_weighting:
        ei  = data.edge_index.to(device)
        deg = torch.zeros(data.num_nodes, dtype=torch.float, device=device)
        deg.scatter_add_(0, ei[0], torch.ones(ei.size(1), device=device))
        deg = deg + 1.0
    protos, counts = {}, {}
    for c in [0, 1]:
        cm = labeled & (labels == c)
        counts[c] = int(cm.sum())
        if counts[c] == 0:
            protos[c] = torch.zeros(embeddings.shape[1], device=device)
            continue
        emb_c = embeddings[cm]
        if cfg.use_degree_weighting:
            w = deg[cm].unsqueeze(1)
            protos[c] = (emb_c * w).sum(0) / w.sum()
        else:
            protos[c] = emb_c.mean(0)
    return protos, counts


def chain_prototype_inheritance(client_emb_protos, client_cls_counts,
                                  clients, rnd, cfg):
    if rnd == 0 or client_emb_protos is None:
        return None
    n = len(client_emb_protos)
    client_protos = {}
    for i in range(n):
        if i == 0 or client_emb_protos[i-1] is None:
            own = client_emb_protos[i]
            client_protos[i] = {
                c: F.normalize(own[c].unsqueeze(0), dim=1).squeeze(0)
                for c in [0, 1]
            }
        else:
            pred = client_emb_protos[i-1]
            own  = client_emb_protos[i]
            blended = {}
            for c in [0, 1]:
                p = F.normalize(pred[c].unsqueeze(0), dim=1).squeeze(0)
                o = F.normalize(own[c].unsqueeze(0), dim=1).squeeze(0)
                alpha = cfg.ema_momentum if c == 1 else 0.3
                blended[c] = F.normalize(
                    (alpha * p + (1 - alpha) * o).unsqueeze(0), dim=1
                ).squeeze(0)
            client_protos[i] = blended
    return client_protos


def supervised_round_client(model, data, client, device, cfg,
                              global_model=None, global_protos=None, lam=0.0,
                              proto_dim=None):
    model.train()
    opt   = Adam(model.parameters(), lr=cfg.lr)
    sched = CosineAnnealingWarmRestarts(opt, T_0=max(1, cfg.sup_epochs // 2), eta_min=cfg.lr_min)
    ei    = get_local_edge_index(data, client['train_mask'], device)
    vmask = client['train_mask'] & (data.y >= 0)
    lbls  = data.y[vmask].to(device)
    enc_ref = (
        {n: p.detach().clone() for n, p in global_model.encoder.named_parameters()}
        if global_model is not None and cfg.use_fedprox else None
    )
    # Guard: BatchNorm1d in train mode with a single sample produces NaN weights.
    # This can happen on small client windows in the sensitivity sweep.
    if vmask.sum() < 2:
        return  # skip training — global weights unchanged for this client
    for _ in range(cfg.sup_epochs):
        opt.zero_grad()
        # BUGFIX: forward the same trunc_dim used by the auxiliary prototype
        # loss below, so the classification head itself is capacity-limited
        # by the tier -- not just the extra prototype regularization term.
        logits, z = model(data.x.to(device), ei, trunc_dim=proto_dim)
        loss = supervised_loss(logits[vmask.to(device)], lbls, device, cfg)
        if cfg.use_fedprox and enc_ref is not None and cfg.mu_encoder > 0:
            enc_prox = sum(
                ((p - enc_ref[n]) ** 2).sum()
                for n, p in model.encoder.named_parameters()
            )
            loss = loss + (cfg.mu_encoder / 2) * enc_prox
        if cfg.use_protos and global_protos is not None and lam > 0:
            # EP-FedProto v3: optional fixed-dim truncation of the transmitted
            # prototype (used by the fixed-budget-per-tier baseline). None =
            # full-dim behaviour, unchanged from v10.
            if proto_dim is not None:
                _protos = {c: v[:proto_dim] for c, v in global_protos.items()}
                _z = z[:, :proto_dim]
            else:
                _protos, _z = global_protos, z
            proto_loss = prototype_supcon_loss(
                _z, data.y, client['train_mask'], _protos, device, cfg
            )
            loss = loss + lam * proto_loss
        loss.backward()
        opt.step()
        sched.step()


def fedper_head_finetune(global_model, data, clients, device, cfg, verbose=True,
                          proto_dim: int = None):
    """
    FedPer: each client fine-tunes its own head locally, encoder frozen.
    FIX: heads are NOT averaged back (v9 bug). Each client keeps its own head.
    Returns (client_results, aggregated_metrics).

    proto_dim: EP-FedProto v3 fixed-tier baseline (BUGFIX). Without this,
    FedPer's head fine-tuning and evaluation silently ran at full emb_dim
    regardless of the tier, undoing the truncation applied everywhere else
    in run_pgfcl -- since use_fedper=True by default, this phase runs for
    every fixed-tier seed and would have overwritten the correctly-truncated
    Phase-2 metrics with full-capacity ones whenever FedPer scored higher.
    """
    if not cfg.use_fedper or cfg.head_finetune_rounds == 0:
        all_train = torch.zeros(data.num_nodes, dtype=torch.bool)
        all_test  = torch.zeros(data.num_nodes, dtype=torch.bool)
        for c in clients:
            all_train |= c['train_mask']
            all_test  |= c['test_mask']
        m = evaluate_tuned(global_model, data, all_train, all_test, device, cfg,
                            proto_dim=proto_dim)
        return [(global_model, m)], m

    if verbose:
        print(f'  [FedPer] Local head FT ({cfg.head_finetune_rounds} rounds, encoder frozen)...')

    client_results = []
    for i, client in enumerate(clients):
        lm = FullGAT(data.num_node_features, cfg).to(device)
        lm.load_state_dict(global_model.state_dict())
        for p in lm.encoder.parameters():
            p.requires_grad_(False)
        lm.train()
        opt   = Adam(lm.head.parameters(), lr=cfg.lr_min * 5)
        sched = CosineAnnealingWarmRestarts(opt, T_0=max(1, cfg.head_finetune_rounds // 2), eta_min=cfg.lr_min)
        ei    = get_local_edge_index(data, client['train_mask'], device)
        vmask = client['train_mask'] & (data.y >= 0)
        lbls  = data.y[vmask].to(device)
        # Total head-only epochs = head_finetune_rounds × sup_epochs
        # (default: 20 × 25 = 500). The outer loop mirrors the federated round
        # structure used during main training; the inner loop is the local SGD.
        for _ in range(cfg.head_finetune_rounds):
            for _ in range(cfg.sup_epochs):
                opt.zero_grad()
                logits, _ = lm(data.x.to(device), ei, trunc_dim=proto_dim)
                supervised_loss(logits[vmask.to(device)], lbls, device, cfg).backward()
                opt.step()
                sched.step()
        for p in lm.encoder.parameters():
            p.requires_grad_(True)
        m = evaluate_tuned(lm, data, client['train_mask'], client['test_mask'], device, cfg,
                            proto_dim=proto_dim)
        if verbose:
            print(f'    Client {i}: F1={m["f1"]:.4f} Prec={m["prec"]:.4f} Rec={m["rec"]:.4f}')
        client_results.append((lm, m))

    agg = avg_metrics([r[1] for r in client_results])
    if verbose:
        print(f'  [FedPer] Agg F1={agg["f1"]:.4f} ± {agg["f1_std"]:.4f}')
    return client_results, agg


print('Training functions defined (v10: in-round SSL removed, FedPer fixed).')


In [ ]:
def run_pgfcl(data: Data, clients: list, device: torch.device,
               cfg: ExperimentConfig, seed: int,
               verbose: bool = True, label: str = 'PGFCL v10',
               proto_dim: int = None):
    """
    PGFCL main training loop.

    v10 changes vs v9:
    - IN-ROUND SSL REMOVED. ssl_pretrain_client is no longer called each round.
      SSL pre-warm still happens (ssl_pretrain_phase), then the main loop is
      purely supervised + prototype aggregation.
      Rationale: Diag-A showed in-round SSL (20 epochs per client per round)
      was net-negative: F1 dropped 3.3 pp vs No-SSL variant.
    - fedper_head_finetune replaces the manual head-averaging block.
      Heads are kept local; never averaged back.
    - config_hash stored in returned metrics for checkpoint integrity checking.

    proto_dim: EP-FedProto v3 -- if set, truncates the transmitted
    prototype (and the local z used against it) to this many dims for
    the ENTIRE run (fixed-tier baseline). None = full emb_dim, v10
    behaviour unchanged.

    Returns (best_metrics, drift_curve, f1_curve, saliency_history).
    """
    set_seed(seed)
    global_model = FullGAT(data.num_node_features, cfg).to(device)
    prev_protos  = None
    drift_curve, f1_curve = [], []
    proto_gen_times = []
    best_f1, best_m = 0., {'acc':0.,'f1':0.,'auc':0.,'prec':0.,'rec':0.,'cm':None}

    all_train = torch.zeros(data.num_nodes, dtype=torch.bool)
    all_test  = torch.zeros(data.num_nodes, dtype=torch.bool)
    for c in clients:
        all_train |= c['train_mask']
        all_test  |= c['test_mask']

    # Phase 1: SSL pre-warm (encoder only, federated, NO supervised signal)
    global_model = ssl_pretrain_phase(global_model, data, clients, device, cfg, verbose)

    sizes    = [c['n_train'] for c in clients]
    ei_full  = data.edge_index.to(device)
    saliency_history = []

    # Phase 2: Main supervised + prototype federated loop (NO in-round SSL)
    for rnd in range(cfg.global_rounds):
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))
        if verbose:
            print(f'\n  === Round {rnd+1}/{cfg.global_rounds} | lam={lam:.3f} ===')

        x = data.x.to(device)
        client_emb_ps, client_counts = [], []

        # Compute per-client prototypes from the current global encoder
        # (no SSL step here — encoder is only updated by supervised + FedAvg)
        _proto_t0 = time.time()
        for j, client in enumerate(clients):
            with torch.no_grad():
                z = global_model.encoder(x, ei_full)
            emb_p, counts = compute_embedding_prototypes(
                z, data, client['train_mask'], device, cfg
            )
            client_emb_ps.append(emb_p)
            client_counts.append(counts)
        proto_gen_times.append(time.time() - _proto_t0)

        # Chain prototype inheritance
        client_protos = None
        global_protos = None
        if cfg.use_protos:
            client_protos = chain_prototype_inheritance(
                client_emb_ps, client_counts, clients, rnd, cfg
            )
            global_protos = client_protos[0] if client_protos else None
            if global_protos is not None and prev_protos is not None:
                d0 = (global_protos[0].float() - prev_protos[0].float()).norm().item()
                d1 = (global_protos[1].float() - prev_protos[1].float()).norm().item()
                drift_curve.append({'round': rnd+1, 'drift_licit': d0, 'drift_illicit': d1})
                if verbose:
                    print(f'  Drift licit={d0:.4f} illicit={d1:.4f}')
            if global_protos is not None:
                prev_protos = {c: v.detach().clone() for c, v in global_protos.items()}

        # Per-client supervised local training
        local_models = []
        for i, client in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg).to(device)
            lm.load_state_dict(global_model.state_dict())
            _cproto = client_protos[i] if (client_protos is not None and lam > 0) else None
            supervised_round_client(lm, data, client, device, cfg,
                global_model=global_model if cfg.use_fedprox else None,
                global_protos=_cproto, lam=lam, proto_dim=proto_dim)
            local_models.append(lm)

        # Aggregation
        if cfg.use_grad_guard:
            agg_weights, _ = grad_guard_weights(global_model, local_models, sizes, cfg)
        elif cfg.use_contrib_agg and global_protos is not None:
            agg_weights = compute_client_contrib_weights(client_emb_ps, sizes, global_protos, cfg)
        else:
            total = sum(sizes)
            agg_weights = [s / total for s in sizes]

        global_model = fedavg_state_contrib(global_model, local_models, agg_weights)

        m = evaluate_tuned(global_model, data, all_train, all_test, device, cfg,
                            proto_dim=proto_dim)
        f1_curve.append({'round': rnd+1, 'f1': m['f1'], 'auc': m['auc']})
        if m['f1'] > best_f1:
            best_f1, best_m = m['f1'], m
        if verbose:
            print(f'  Global | F1={m["f1"]:.4f} | AUC={m["auc"]:.4f} | '
                  f'Prec={m["prec"]:.4f} | Rec={m["rec"]:.4f} | thresh={m["thresh"]:.2f}')

        if cfg.use_saliency and (rnd + 1) % 10 == 0:
            sal = extract_node_saliency(global_model, data, all_test, device, cfg)
            saliency_history.append({'round': rnd+1, **sal})
            if verbose:
                n_correct = int((sal['true_labels'] == 1).sum())
                print(f'  Saliency top-{cfg.saliency_top_k}: ' +
                      f'{n_correct}/{cfg.saliency_top_k} are truly illicit')

    # Phase 3: FedPer head fine-tuning (local, no averaging of heads)
    if cfg.head_finetune_rounds > 0:
        client_results, per_results = fedper_head_finetune(
            global_model, data, clients, device, cfg, verbose=verbose,
            proto_dim=proto_dim
        )
        # Report per-client aggregated metrics; use best model from Phase 2 if FedPer is worse
        if per_results['f1'] > best_f1:
            best_f1, best_m = per_results['f1'], per_results
            best_m['fedper_client_results'] = [(None, r[1]) for r in client_results]
        if verbose:
            print(f'  [FedPer] Final agg F1={per_results["f1"]:.4f}')

    # ECE calibration on global model
    if cfg.use_calibration:
        # Pass all_train so compute_ece uses inductive edges (same as evaluate_tuned)
        ece, bin_data = compute_ece(global_model, data, all_test, device, cfg,
                                    train_mask=all_train, proto_dim=proto_dim)
        best_m['ece']      = ece
        best_m['ece_bins'] = bin_data
        if verbose:
            print(f'  ECE = {ece:.4f}')

    # EP-FedProto v3: instrumentation -- mean per-round prototype-generation time
    best_m['proto_gen_time_s'] = float(np.mean(proto_gen_times)) if proto_gen_times else 0.0
    best_m['proto_dim'] = proto_dim if proto_dim is not None else cfg.emb_dim

    # Store config hash for checkpoint integrity
    best_m['config_hash'] = config_hash(cfg)

    print(f'\n[{label}] Best F1={best_m["f1"]:.4f} | AUC={best_m["auc"]:.4f} | ' +
          f'hash={best_m["config_hash"]}')
    return best_m, drift_curve, f1_curve, saliency_history


print('run_pgfcl v10 defined.')


## EP-FedProto v3 — Shared Infra (from NB1: device tiers, multi-budget loss, instrumentation)

In [ ]:
# ── EP-FedProto v3: device tiers, multi-budget loss, instrumentation ─────────
# NB1 scope per plan: multi-budget loss (confirm single-pass slicing),
# fixed-tier baseline, FjORD baseline, edge-device config, instrumentation,
# scalability-sweep setup. This cell defines shared infra used by all of them;
# the actual new baseline runs happen further below, after "Baselines".

DEVICE_TIER_DIMS = (8, 16, 32, 64)   # nested prototype/embedding budgets


def assign_device_tiers(clients, tier_dims=DEVICE_TIER_DIMS, distribution=None, seed=0):
    """Assign each client a device-capability tier (max transmitted dim).

    distribution: optional list of proportions matching tier_dims (sums to 1).
    Default: uniform round-robin, shuffled. Used by FjORD (heterogeneous
    per-client widths) now, and by NB2's device-tier lookup / NB3's skewed
    60/30/10 tier-distribution ablation later.
    """
    rng = random.Random(seed)
    n = len(clients)
    if distribution is None:
        tiers = [tier_dims[i % len(tier_dims)] for i in range(n)]
    else:
        assert len(distribution) == len(tier_dims), "distribution must match tier_dims"
        counts = [round(p * n) for p in distribution]
        while sum(counts) < n: counts[counts.index(min(counts))] += 1
        while sum(counts) > n: counts[counts.index(max(counts))] -= 1
        tiers = []
        for d, c in zip(tier_dims, counts):
            tiers += [d] * c
    rng.shuffle(tiers)
    return {c['id']: t for c, t in zip(clients, tiers)}


def truncate_proto(proto_dict, d):
    """Slice each class prototype to its first d dims (Matryoshka nesting)."""
    return {c: v[:d] for c, v in proto_dict.items()}


def multi_budget_proto_loss(z, labels, mask, global_protos_full, device, cfg,
                             budgets=DEVICE_TIER_DIMS, weights=None):
    """Matryoshka-style multi-budget prototype loss.

    SINGLE-PASS SLICING (confirmed by unit test): z is encoded ONCE by the
    caller; this function only slices *columns* of that same tensor per
    budget -- it never re-runs the encoder. Cost scales as O(len(budgets))
    prototype-loss evaluations on views of one tensor, not O(len(budgets))
    forward passes.

    Verified in isolation before wiring in: gradient flows into every
    budget's slice, and per-dim gradient magnitude decreases monotonically
    from "used by all 4 tiers" (dims 0:8) to "used only by the d=64 tier"
    (dims 32:64) -- exactly the nesting behaviour Matryoshka representations
    are supposed to have.

    Not yet wired into run_pgfcl's main loop -- that happens in NB2, where
    EP-FedProto's full training loop (this loss + the rank-aware aggregator)
    is assembled. This cell only defines and validates the loss itself.
    """
    if global_protos_full is None:
        return torch.tensor(0.0, device=device)
    if weights is None:
        weights = [1.0 / len(budgets)] * len(budgets)
    assert len(weights) == len(budgets)
    total = torch.tensor(0.0, device=device)
    for w, d in zip(weights, budgets):
        protos_d = truncate_proto(global_protos_full, d)
        total = total + w * prototype_supcon_loss(
            z[:, :d], labels, mask, protos_d, device, cfg
        )
    return total


# ── Instrumentation ───────────────────────────────────────────────────────────
try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except ImportError:
    psutil = None
    _PROC = None
    print('  [instrumentation] psutil not available -- CPU memory stats will be skipped.')


def with_instrumentation(fn):
    """Wrap a run_*_gat baseline: records wall time + peak GPU/CPU memory into
    the returned metrics dict under 'compute_stats'. Only fires when fn() is
    actually executed -- a run_or_load cache hit correctly reports no new
    compute stats (nothing ran)."""
    def wrapper(*args, **kwargs):
        device = kwargs.get('device')
        if device is None:
            for a in args:
                if isinstance(a, torch.device):
                    device = a
                    break
        if device is not None and device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats(device)
        t0 = time.time()
        result = fn(*args, **kwargs)
        stats = {'wall_time_s': time.time() - t0}
        if device is not None and device.type == 'cuda':
            stats['peak_gpu_mem_mb'] = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
        if _PROC is not None:
            stats['peak_cpu_mem_mb'] = _PROC.memory_info().rss / (1024 ** 2)
        if isinstance(result, dict):
            result['compute_stats'] = stats
        return result
    return wrapper


print('EP-FedProto v3 infra defined: device tiers, multi-budget loss, instrumentation.')


## Baseline Runners Needed For the Core/Ablation Comparisons

Only `run_fedproto_gat` (plain FedProto: `run_pgfcl` with SSL/FedProx off,
prototypes on) and `run_fjord_gat` (Ordered Dropout) are reused here — the
other NB1 baselines (central/local-only/FedAvg/FedSage/MOON/SCAFFOLD) are only
needed at the default client count and already live in the bundle chain.

In [ ]:
def _all_masks(clients, n_nodes):
    all_train = torch.zeros(n_nodes, dtype=torch.bool)
    all_test  = torch.zeros(n_nodes, dtype=torch.bool)
    for c in clients:
        all_train |= c['train_mask']
        all_test  |= c['test_mask']
    return all_train, all_test


def run_fedproto_gat(data, clients, device, cfg, seed=None):
    """Verbatim from NB1: proto_cfg is always fresh ExperimentConfig() defaults
    (SSL off, prototypes on, FedProx off) regardless of the `cfg` passed in --
    this is a pre-existing property of the baseline, kept as-is for fidelity."""
    seed = seed or cfg.seeds[0]
    print(f'  [FedProto GAT] seed={seed}')
    proto_cfg = ExperimentConfig(use_ssl=False, use_protos=True, use_fedprox=False)
    m, _, f1_curve, _ = run_pgfcl(data, clients, device, proto_cfg, seed=seed,
                         verbose=False, label='FedProto GAT')
    # SCALABILITY-SWEEP FIX: retain the per-round F1 curve run_pgfcl already
    # computes (previously discarded via the `_` above), plus the emb_dim and
    # client count actually used for THIS run (proto_cfg's own emb_dim, which
    # may differ from whatever `cfg` the caller passed in) -- so comm bytes
    # can be computed per round from what actually ran, not assumed from CFG.
    m['f1_curve']  = f1_curve
    m['comm_dim']  = proto_cfg.emb_dim
    m['n_clients'] = len(clients)
    print(f'    [FedProto GAT] F1={m["f1"]:.4f} | AUC={m["auc"]:.4f}')
    return m


_FEDPROTO_EXPECTED_HASH = config_hash(ExperimentConfig(use_ssl=False, use_protos=True, use_fedprox=False))
print('run_fedproto_gat ready.')


## FjORD (Ordered Dropout) — full federated baseline (from NB1/NB2)

In [ ]:
# ── FjORD (Ordered Dropout) baseline ──────────────────────────────────────────
def evaluate_at_width(model, data, train_mask, test_mask, device, cfg, width_ratio):
    """FjORD per-width evaluation (fills the 'no per-tier eval yet' gap).

    Mirrors evaluate_tuned's threshold-tuned, inductive-eval logic exactly,
    but forwards through encoder.forward_od(width_ratio) instead of the
    full-width encoder.forward(), so FjORD gets a real F1-per-width number
    -- comparable to the fixed-tier FedProto and EP-FedProto F1-vs-d curves
    -- instead of only a single full-capacity ceiling F1 per run.
    """
    model.eval()
    ei_tr = get_local_edge_index(data, train_mask, device)
    with torch.no_grad():
        emb_tr    = model.encoder.forward_od(data.x.to(device), ei_tr, width_ratio)
        logits_tr = model.head(emb_tr)
    probs_tr = F.softmax(logits_tr, dim=1)[:, 1].cpu().numpy()
    if not np.isfinite(probs_tr).all():
        return {'acc': 0., 'f1': 0., 'auc': 0., 'prec': 0., 'rec': 0.,
                'cm': np.zeros((2, 2), int), 'thresh': 0.5}

    tr_lab = train_mask & (data.y >= 0)
    best_thresh = 0.5
    if tr_lab.sum() > 0 and len(np.unique(data.y[tr_lab].numpy())) > 1:
        p, r, thresholds = precision_recall_curve(
            data.y[tr_lab].numpy(), probs_tr[tr_lab.numpy()])
        f1s = 2 * p * r / (p + r + 1e-8)
        best_thresh = float(np.clip(thresholds[np.argmax(f1s[:-1])], 0.1, 0.9))

    ei_te = get_inductive_edge_index(data, train_mask, test_mask, device)
    with torch.no_grad():
        emb_te    = model.encoder.forward_od(data.x.to(device), ei_te, width_ratio)
        logits_te = model.head(emb_te)
    probs_te_full = F.softmax(logits_te, dim=1)[:, 1]
    if cfg.use_label_prop:
        probs_te_full = label_propagation(
            probs_te_full, ei_te, data.num_nodes,
            alpha=cfg.lp_alpha, steps=cfg.lp_steps
        ).to(device)
    probs_te = probs_te_full.cpu().numpy()

    te_lab = test_mask & (data.y >= 0)
    if te_lab.sum() == 0:
        return {'acc': 0., 'f1': 0., 'auc': 0., 'prec': 0., 'rec': 0.,
                'cm': np.zeros((2, 2), int), 'thresh': best_thresh}

    probs_te_masked = probs_te[te_lab.numpy()]
    preds_te        = (probs_te_masked >= best_thresh).astype(int)
    true_te         = data.y[te_lab].numpy()

    return {
        'acc':    accuracy_score(true_te, preds_te),
        'f1':     f1_score(true_te, preds_te, zero_division=0),
        'auc':    roc_auc_score(true_te, probs_te_masked) if len(np.unique(true_te)) > 1 else 0.,
        'prec':   precision_score(true_te, preds_te, zero_division=0),
        'rec':    recall_score(true_te, preds_te, zero_division=0),
        'cm':     confusion_matrix(true_te, preds_te),
        'thresh': best_thresh
    }


def run_fjord_gat(data, clients, device, cfg, seed=None,
                   tier_dims=DEVICE_TIER_DIMS, tier_distribution=None):
    """FjORD: weight-space heterogeneity via Ordered Dropout, contrasted in
    the paper's framing against EP-FedProto's prototype-space nesting. Each
    client gets a device tier (max active embedding width) and trains only
    that nested channel-prefix of the shared encoder (encoder.forward_od);
    aggregation is plain size-weighted FedAvg over the full state_dict.

    This is correct (not just convenient) because Ordered Dropout here is
    post-activation channel MASKING, not weight resizing: a masked-out output
    channel gets exactly zero gradient through the weight rows that produce
    it (verified in isolation), so a low-budget client's local weights for
    channels above its tier are simply left equal to the global value it
    started the round with -- averaging them in is a no-op for those rows,
    not noise from an untrained client.

    No prototypes, no FedProx -- pure weight-space nesting baseline.
    """
    seed = seed or cfg.seeds[0]
    set_seed(seed)
    tiers = assign_device_tiers(clients, tier_dims, tier_distribution, seed=seed)
    print(f'  [FjORD] seed={seed} | tiers={tiers}')

    global_model = FullGAT(data.num_node_features, cfg).to(device)
    sizes = [c['n_train'] for c in clients]
    all_train, all_test = _all_masks(clients, data.num_nodes)
    best_f1, best_m = 0., {}
    f1_curve = []  # SCALABILITY-SWEEP FIX: per-round F1/AUC, previously discarded

    for rnd in range(cfg.baseline_rounds):
        local_models = []
        for i, client in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg).to(device)
            lm.load_state_dict(global_model.state_dict())
            lm.train()
            opt = Adam(lm.parameters(), lr=cfg.lr)
            ei = get_local_edge_index(data, client['train_mask'], device)
            vmask = client['train_mask'] & (data.y >= 0)
            if vmask.sum() < 2:
                local_models.append(lm)
                continue
            lbls = data.y[vmask].to(device)
            width_ratio = tiers[client['id']] / cfg.emb_dim
            for _ in range(cfg.baseline_local_epochs):
                opt.zero_grad()
                emb = lm.encoder.forward_od(data.x.to(device), ei, width_ratio)
                logits = lm.head(emb)
                loss = supervised_loss(logits[vmask.to(device)], lbls, device, cfg)
                loss.backward()
                opt.step()
            local_models.append(lm)
        global_model = fedavg_state_selective(global_model, local_models, sizes)
        m = evaluate_tuned(global_model, data, all_train, all_test, device, cfg)
        f1_curve.append({'round': rnd + 1, 'f1': m['f1'], 'auc': m['auc']})
        if m['f1'] > best_f1:
            best_f1, best_m = m['f1'], m

    # GAP FIX: per-width F1 curve on the FINAL global model (intermediate
    # per-round models aren't retained, so this is evaluated post-hoc on the
    # model state at the end of training, same as e.g. run_centralized_gat
    # does for its single reported number -- not on whichever round produced
    # best_f1 at full width).
    best_m['f1_by_width'] = {
        d: evaluate_at_width(global_model, data, all_train, all_test, device, cfg,
                              d / cfg.emb_dim)
        for d in tier_dims
    }
    best_m['device_tiers'] = tiers
    best_m['config_hash']  = config_hash(cfg)

    # SCALABILITY-SWEEP FIX: retain the per-round F1 curve (previously only
    # best_f1/best_m survived past the loop). No prototype communication by
    # design (pure weight-space method), so comm_curve is all zeros -- kept
    # in the same shape as EP-FedProto/FedProto's comm_curve so downstream
    # code (e.g. the scalability sweep) can treat all three methods uniformly.
    best_m['f1_curve']          = f1_curve
    best_m['comm_curve']        = [{'round': c['round'], 'proto_bytes': 0} for c in f1_curve]
    best_m['total_proto_bytes'] = 0
    best_m['n_clients']         = len(clients)

    print(f'  [FjORD] Best F1={best_m["f1"]:.4f} | AUC={best_m["auc"]:.4f}')
    for d in tier_dims:
        fw = best_m['f1_by_width'][d]
        print(f'    [FjORD] width d={d:2d} (ratio={d/cfg.emb_dim:.2f}): F1={fw["f1"]:.4f} | AUC={fw["auc"]:.4f}')
    return best_m

run_fjord_gat = with_instrumentation(run_fjord_gat)
print('run_fjord_gat defined.')


## Rank-Aware ContribAgg Aggregator, Dimension-Wise (from NB2)

Standard ContribAgg (`compute_client_contrib_weights` / `chain_prototype_inheritance`,
used by plain FedProto above) weights each client's *whole* prototype update by a
single cosine-similarity-to-consensus quality score. That's not well-defined for
nested prototypes: a client whose tier is `d=8` has no opinion at all about
dimensions `[8:64]`. The rank-aware aggregator instead aggregates the global
prototype **segment-by-segment** along the dimension axis, using `DEVICE_TIER_DIMS`
as the segment boundaries `[0,8) [8,16) [16,32) [32,64)`:

- segment `[0,8)`   — every client contributes (every tier includes the first 8 dims)
- segment `[8,16)`  — only clients with tier `>= 16` contribute
- segment `[16,32)` — only clients with tier `>= 32` contribute
- segment `[32,64)` — only clients with tier `== 64` contribute

Within a segment, weight is the same ContribAgg quality score used elsewhere
(cosine similarity of that client's local segment to the previous global
segment, floored via `cfg.contrib_floor`, size-weighted), renormalized over
only the eligible subset. Concatenating the aggregated segments yields a new
global prototype that is nested by construction: `truncate_proto(global, d)`
is exactly the vector every client of tier `>= d` was trained against.

`uniform_dimwise_agg` (also from NB2) is the same segment-eligibility rule
with plain size-only weighting (no ContribAgg quality term) — this notebook's
`Uniform-Agg` ablation arm.

In [ ]:
def rank_aware_contrib_agg(client_full_protos: list, client_tier_dims: list,
                            client_sizes: list, prev_global_full_protos,
                            cfg: ExperimentConfig, tier_dims=DEVICE_TIER_DIMS) -> dict:
    """
    client_full_protos : list of {0: vec[emb_dim], 1: vec[emb_dim]} — each
                          client's own FULL-width local prototype (computed the
                          same way as for plain FedProto, via
                          compute_embedding_prototypes). Only the segments up to
                          that client's own tier are ever used below; a client
                          never contributes to a segment its tier doesn't cover.
    client_tier_dims   : list[int] — each client's device-tier budget (see
                          assign_device_tiers), aligned by index with
                          client_full_protos / client_sizes.
    Returns a full nested global-prototype dict {0: vec[emb_dim], 1: vec[emb_dim]}
    built by concatenating dimension-wise-aggregated segments -- pass this
    straight into multi_budget_proto_loss / truncate_proto.
    """
    boundaries = [0] + list(tier_dims)   # [0, 8, 16, 32, 64]
    n = len(client_full_protos)
    seg_out = {0: [], 1: []}

    for seg_lo, seg_hi in zip(boundaries[:-1], boundaries[1:]):
        eligible = [i for i in range(n) if client_tier_dims[i] >= seg_hi]
        if not eligible:
            max_dim = max(client_tier_dims)
            eligible = [i for i in range(n) if client_tier_dims[i] == max_dim]

        weights = []
        for i in eligible:
            sz = client_sizes[i]
            if prev_global_full_protos is None:
                weights.append(sz ** 0.5)
                continue
            local_seg  = client_full_protos[i][1][seg_lo:seg_hi]
            global_seg = prev_global_full_protos[1][seg_lo:seg_hi]
            if local_seg.norm() < 1e-8 or global_seg.norm() < 1e-8:
                quality = 1.0
            else:
                cos_sim = F.cosine_similarity(local_seg.unsqueeze(0),
                                               global_seg.unsqueeze(0)).item()
                quality = cfg.contrib_floor + (1.0 - cfg.contrib_floor) * (cos_sim + 1.0) / 2.0
            weights.append((sz ** 0.5) * quality)
        total_w = sum(weights) + 1e-8

        for c in [0, 1]:
            acc = torch.zeros(seg_hi - seg_lo, device=client_full_protos[0][c].device)
            for i, w in zip(eligible, weights):
                acc += (w / total_w) * client_full_protos[i][c][seg_lo:seg_hi]
            seg_out[c].append(acc)

    return {c: torch.cat(seg_out[c], dim=0) for c in [0, 1]}


def uniform_dimwise_agg(client_full_protos: list, client_tier_dims: list,
                         client_sizes: list, prev_global_full_protos=None,
                         cfg=None, tier_dims=DEVICE_TIER_DIMS) -> dict:
    """Ablation arm 'Uniform-Agg' (NB3): same segment-eligibility rule as
    rank_aware_contrib_agg, but size-only weighting (no ContribAgg quality term).
    Accepts (and ignores) prev_global_full_protos/cfg so it is a drop-in
    agg_fn for run_ep_fedproto, which always calls agg_fn(..., prev_protos,
    cfg, tier_dims=tier_dims) regardless of which aggregator is plugged in."""
    boundaries = [0] + list(tier_dims)
    n = len(client_full_protos)
    seg_out = {0: [], 1: []}
    for seg_lo, seg_hi in zip(boundaries[:-1], boundaries[1:]):
        eligible = [i for i in range(n) if client_tier_dims[i] >= seg_hi]
        if not eligible:
            max_dim = max(client_tier_dims)
            eligible = [i for i in range(n) if client_tier_dims[i] == max_dim]
        total_sz = sum(client_sizes[i] for i in eligible)
        for c in [0, 1]:
            acc = torch.zeros(seg_hi - seg_lo, device=client_full_protos[0][c].device)
            for i in eligible:
                acc += (client_sizes[i] / total_sz) * client_full_protos[i][c][seg_lo:seg_hi]
            seg_out[c].append(acc)
    return {c: torch.cat(seg_out[c], dim=0) for c in [0, 1]}


print('Rank-aware dimension-wise ContribAgg aggregator defined.')


## Communication-Byte Accounting (from NB2, feeds NB4's headline number / Pareto figure)

Encoder+head are FedAvg'd in full every round for **all** of FedProto, FjORD and
EP-FedProto here (nesting lives in the prototype, not the weights, per the
paper's framing) — that full-model term is therefore a constant shared across
methods and is **not** where any communication reduction comes from. The
`proto_bytes` term below is the actual differentiator: FedProto always sends
the full `emb_dim`-wide prototype; EP-FedProto sends each client's
tier-truncated nested prototype. FjORD, being a weight-space method, doesn't
transmit a prototype at all — its efficiency story here is a **compute**
story (Ordered Dropout reduces each client's active-channel FLOPs), not a
communication one; it's reported with `proto_bytes = 0` for an honest,
consistent Pareto x-axis rather than an invented savings number.

In [ ]:
FLOAT_BYTES = 4  # fp32
N_CLASSES = 2

def encoder_head_param_bytes(data, cfg) -> int:
    m = FullGAT(data.num_node_features, cfg)
    n_params = sum(p.numel() for p in m.parameters())
    del m
    return n_params * FLOAT_BYTES


def round_proto_bytes_uniform(dim: int, n_clients: int) -> int:
    """FedProto (fixed dim for everyone): upload (n_clients x dim) + broadcast
    (n_clients x dim) of the aggregated global prototype, per class."""
    return (n_clients * dim * N_CLASSES * FLOAT_BYTES) * 2


def round_proto_bytes_nested(client_tier_dims: list) -> int:
    """EP-FedProto: each client uploads only its own tier-dim slice; the
    server broadcasts back the (nested) global prototype truncated to each
    client's own tier -- nobody ever receives more than they can use."""
    upload   = sum(d * N_CLASSES * FLOAT_BYTES for d in client_tier_dims)
    download = sum(d * N_CLASSES * FLOAT_BYTES for d in client_tier_dims)
    return upload + download


print('Comm-byte accounting functions ready.')


## Main EP-FedProto Training Loop (from NB2, extended in this notebook with a `nested` flag)

Structurally mirrors `run_pgfcl` (SSL pre-warm -> per-round supervised local
training + prototype aggregation -> FedPer head fine-tune -> ECE), but:

- prototypes are computed at full width every round (`compute_embedding_prototypes`,
  unchanged) — nesting only happens at aggregation/loss time, never by
  re-encoding
- the auxiliary loss is `multi_budget_proto_loss` (from NB1, equal-weighted
  across every nesting level up to the client's own tier) instead of the
  single-dim `prototype_supcon_loss`
- aggregation of the prototype uses `rank_aware_contrib_agg` by default
  (dimension-wise, tier-eligibility-aware) instead of
  `chain_prototype_inheritance` + `compute_client_contrib_weights` — but is
  now pluggable via `agg_fn` (NB3: `uniform_dimwise_agg` for the Uniform-Agg
  ablation)
- the classifier forward pass is **never truncated** (`trunc_dim=None`
  throughout) — nesting lives in the prototype, not the weights; encoder+head
  are FedAvg'd in full via plain `fedavg_state_selective`, exactly like FjORD
  and FedProto
- per-round prototype-communication bytes are logged
  (`round_proto_bytes_nested`)
- each client's device tier is drawn once per run via `assign_device_tiers`
  (reused verbatim from NB1), consistent with the device-tier -> dimension
  lookup design
- **NB3 addition:** a `nested` flag (default `True`, fully backward-compatible
  with NB2's cached checkpoints) controls whether `supervised_round_client_ep`
  trains against every budget a client's tier covers (the Matryoshka
  multi-budget loss — EP-FedProto's core novelty) or only that client's own
  single tier dimension (the `No-Nesting` ablation).

In [ ]:
def supervised_round_client_ep(model, data, client, device, cfg,
                                global_model=None, global_protos_full=None,
                                lam=0.0, client_tier_dim=None,
                                tier_dims=DEVICE_TIER_DIMS, nested=True):
    """EP-FedProto local training step. Same structure as
    supervised_round_client, but the auxiliary loss is multi_budget_proto_loss
    restricted to the nesting levels this client's own tier can see, and the
    forward pass is never truncated (trunc_dim always None).

    nested (NB3 new work -- 'No-Nesting' ablation): when True (default,
    unchanged from NB2), a client trains against EVERY budget its tier
    covers (tiers_dims <= client_tier_dim) -- this is the Matryoshka
    multi-budget loss that is EP-FedProto's core novelty. When False, a
    client trains against ONLY its own tier's single dimension, exactly
    like the fixed-tier FedProto baseline's loss but still combined with
    the rank-aware dimension-wise aggregator -- isolating what the nested
    loss itself contributes, independent of the aggregator."""
    model.train()
    opt   = Adam(model.parameters(), lr=cfg.lr)
    sched = CosineAnnealingWarmRestarts(opt, T_0=max(1, cfg.sup_epochs // 2), eta_min=cfg.lr_min)
    ei    = get_local_edge_index(data, client['train_mask'], device)
    vmask = client['train_mask'] & (data.y >= 0)
    lbls  = data.y[vmask].to(device)
    enc_ref = (
        {n: p.detach().clone() for n, p in global_model.encoder.named_parameters()}
        if global_model is not None and cfg.use_fedprox else None
    )
    if vmask.sum() < 2:
        return
    if nested:
        own_budgets = tuple(d for d in tier_dims if d <= client_tier_dim) or (tier_dims[0],)
    else:
        # No-Nesting ablation: single-budget loss at exactly this client's tier.
        own_budgets = (client_tier_dim,) if client_tier_dim is not None else (tier_dims[0],)
    own_weights = [1.0 / len(own_budgets)] * len(own_budgets)
    for _ in range(cfg.sup_epochs):
        opt.zero_grad()
        logits, z = model(data.x.to(device), ei, trunc_dim=None)
        loss = supervised_loss(logits[vmask.to(device)], lbls, device, cfg)
        if cfg.use_fedprox and enc_ref is not None and cfg.mu_encoder > 0:
            enc_prox = sum(((p - enc_ref[n]) ** 2).sum()
                           for n, p in model.encoder.named_parameters())
            loss = loss + (cfg.mu_encoder / 2) * enc_prox
        if cfg.use_protos and global_protos_full is not None and lam > 0:
            proto_loss = multi_budget_proto_loss(
                z, data.y, client['train_mask'], global_protos_full, device, cfg,
                budgets=own_budgets, weights=own_weights
            )
            loss = loss + lam * proto_loss
        loss.backward()
        opt.step()
        sched.step()


def run_ep_fedproto(data: Data, clients: list, device: torch.device,
                     cfg: ExperimentConfig, seed: int,
                     tier_dims=DEVICE_TIER_DIMS, tier_distribution=None,
                     agg_fn=None, verbose: bool = True, label: str = 'EP-FedProto',
                     nested: bool = True):
    """Returns (best_metrics, drift_curve, f1_curve, comm_curve, saliency_history).

    nested (NB3 new work): forwarded to supervised_round_client_ep -- see its
    docstring. Default True reproduces NB2's headline run byte-for-byte
    (signature-compatible; existing nb2_bundle cached results remain valid).
    agg_fn default (rank_aware_contrib_agg) and this default (nested=True)
    together ARE 'Full-EP-FedProto' in NB3's 3-new-arms ablation."""
    if agg_fn is None:
        agg_fn = rank_aware_contrib_agg
    set_seed(seed)
    global_model = FullGAT(data.num_node_features, cfg).to(device)
    device_tier_map = assign_device_tiers(clients, tier_dims, tier_distribution, seed=seed)
    client_tier_dims = [device_tier_map[c['id']] for c in clients]
    if verbose:
        print(f'  [EP-FedProto] device tiers: {device_tier_map}')

    prev_protos  = None
    drift_curve, f1_curve, comm_curve = [], [], []
    proto_gen_times = []
    best_f1, best_m = 0., {'acc':0.,'f1':0.,'auc':0.,'prec':0.,'rec':0.,'cm':None}

    all_train, all_test = _all_masks(clients, data.num_nodes)

    global_model = ssl_pretrain_phase(global_model, data, clients, device, cfg, verbose)

    sizes    = [c['n_train'] for c in clients]
    ei_full  = data.edge_index.to(device)
    saliency_history = []

    for rnd in range(cfg.global_rounds):
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))
        if verbose:
            print(f'\n  === Round {rnd+1}/{cfg.global_rounds} | lam={lam:.3f} ===')

        x = data.x.to(device)
        client_full_protos, client_counts = [], []
        _proto_t0 = time.time()
        for client in clients:
            with torch.no_grad():
                z = global_model.encoder(x, ei_full)
            full_p, counts = compute_embedding_prototypes(z, data, client['train_mask'], device, cfg)
            client_full_protos.append(full_p)
            client_counts.append(counts)
        proto_gen_times.append(time.time() - _proto_t0)

        global_protos = None
        if cfg.use_protos:
            global_protos = agg_fn(client_full_protos, client_tier_dims, sizes,
                                    prev_protos, cfg, tier_dims=tier_dims)
            if prev_protos is not None:
                d0 = (global_protos[0].float() - prev_protos[0].float()).norm().item()
                d1 = (global_protos[1].float() - prev_protos[1].float()).norm().item()
                drift_curve.append({'round': rnd+1, 'drift_licit': d0, 'drift_illicit': d1})
            prev_protos = {c: v.detach().clone() for c, v in global_protos.items()}

        local_models = []
        for i, client in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg).to(device)
            lm.load_state_dict(global_model.state_dict())
            supervised_round_client_ep(
                lm, data, client, device, cfg,
                global_model=global_model if cfg.use_fedprox else None,
                global_protos_full=global_protos, lam=lam,
                client_tier_dim=client_tier_dims[i], tier_dims=tier_dims,
                nested=nested
            )
            local_models.append(lm)

        # Encoder+head FedAvg'd in FULL (plain, size-weighted) -- nesting lives
        # in the prototype above, not in the weights.
        global_model = fedavg_state_selective(global_model, local_models, sizes)

        comm_bytes = round_proto_bytes_nested(client_tier_dims)
        comm_curve.append({'round': rnd+1, 'proto_bytes': comm_bytes})

        m = evaluate_tuned(global_model, data, all_train, all_test, device, cfg, proto_dim=None)
        f1_curve.append({'round': rnd+1, 'f1': m['f1'], 'auc': m['auc']})
        if m['f1'] > best_f1:
            best_f1, best_m = m['f1'], m
        if verbose:
            print(f'  Global | F1={m["f1"]:.4f} | AUC={m["auc"]:.4f} | '
                  f'proto_comm={comm_bytes/1024:.2f} KB')

        if cfg.use_saliency and (rnd + 1) % 10 == 0:
            sal = extract_node_saliency(global_model, data, all_test, device, cfg)
            saliency_history.append({'round': rnd+1, **sal})

    if cfg.head_finetune_rounds > 0:
        client_results, per_results = fedper_head_finetune(
            global_model, data, clients, device, cfg, verbose=verbose, proto_dim=None
        )
        if per_results['f1'] > best_f1:
            best_f1, best_m = per_results['f1'], per_results
            best_m['fedper_client_results'] = [(None, r[1]) for r in client_results]

    if cfg.use_calibration:
        ece, bin_data = compute_ece(global_model, data, all_test, device, cfg,
                                     train_mask=all_train, proto_dim=None)
        best_m['ece']      = ece
        best_m['ece_bins'] = bin_data

    best_m['proto_gen_time_s'] = float(np.mean(proto_gen_times)) if proto_gen_times else 0.0
    best_m['config_hash']      = config_hash(cfg)
    best_m['device_tier_map']  = dict(device_tier_map)
    best_m['tier_dims']        = tuple(tier_dims)
    best_m['total_proto_bytes'] = sum(c['proto_bytes'] for c in comm_curve)
    best_m['nested']            = nested  # NB3: bookkeeping for the No-Nesting ablation

    print(f'\n[{label}] Best F1={best_m["f1"]:.4f} | AUC={best_m["auc"]:.4f} | '
          f'total proto comm={best_m["total_proto_bytes"]/1024:.1f} KB')
    return best_m, drift_curve, f1_curve, comm_curve, saliency_history


print('run_ep_fedproto defined.')


## Load NB2 Bundle

In [ ]:
bundle = load_ckpt('nb2_bundle')
if bundle is None:
    raise RuntimeError('Could not find nb2_bundle. Add the NB2 (ep_fedproto_nb2_v2) output as input data.')

# ── carried over from NB1 (unchanged baselines + infra) ──
central_avg, central_runs         = bundle['central_avg'], bundle['central_runs']
localonly_avg, localonly_runs     = bundle['localonly_avg'], bundle['localonly_runs']
fedavg_avg, fedavg_runs           = bundle['fedavg_avg'], bundle['fedavg_runs']
fedsage_avg, fedsage_runs         = bundle['fedsage_avg'], bundle['fedsage_runs']
fedsage_gat_avg, fedsage_gat_runs = bundle['fedsage_gat_avg'], bundle['fedsage_gat_runs']
fedproto_avg, fedproto_runs       = bundle['fedproto_avg'], bundle['fedproto_runs']
moon_avg, moon_runs               = bundle.get('moon_avg', {}), bundle.get('moon_runs', [])
scaffold_avg, scaffold_runs       = bundle.get('scaffold_avg', {}), bundle.get('scaffold_runs', [])
fixed_tier_avg, fixed_tier_runs   = bundle['fixed_tier_avg'], bundle['fixed_tier_runs']
edge_confirmation_run             = bundle['edge_confirmation_run']
device_tier_dims_nb                = bundle['device_tier_dims']
edge_tier_dims                    = bundle['edge_tier_dims']
scalability_n_clients             = bundle['scalability_n_clients']

assert tuple(device_tier_dims_nb) == DEVICE_TIER_DIMS, \
    f"NB2 device_tier_dims {device_tier_dims_nb} must match this notebook's DEVICE_TIER_DIMS {DEVICE_TIER_DIMS}"

# ── NB2 new work: EP-FedProto headline run + friends ──
ep_fedproto_avg          = bundle['ep_fedproto_avg']
ep_fedproto_runs         = bundle['ep_fedproto_runs']
ep_fedproto_drift_all    = bundle['ep_fedproto_drift_all']
ep_fedproto_f1c_all      = bundle['ep_fedproto_f1c_all']
ep_fedproto_comm_all     = bundle['ep_fedproto_comm_all']
ep_fedproto_saliency_all = bundle['ep_fedproto_saliency_all']

fjord_federated_avg, fjord_federated_runs = bundle['fjord_federated_avg'], bundle['fjord_federated_runs']
edge_full_avg, edge_full_runs             = bundle['edge_full_avg'], bundle['edge_full_runs']
edge_cfg_hash                             = bundle['edge_cfg_hash']
scalability_results                       = bundle['scalability_results']

# MAIN_SEEDS: the seeds NB2's headline EP-FedProto / FjORD-federated runs used
# (CFG.seeds[:3]) -- inferred from how many runs are in the bundle rather than
# hardcoded, so this notebook stays correct even if NB2's seed budget changes.
MAIN_SEEDS = CFG.seeds[:len(ep_fedproto_runs)]
assert len(ep_fedproto_runs) == len(fjord_federated_runs) == len(MAIN_SEEDS), (
    'ep_fedproto_runs / fjord_federated_runs seed counts disagree -- '
    'core-comparison pairing below assumes they were run on the same seeds.')

print('NB2 bundle loaded:')
for key, name in [('central_avg', 'central'), ('localonly_avg', 'localonly'),
                   ('fedavg_avg', 'fedavg'), ('fedproto_avg', 'fedproto'),
                   ('ep_fedproto_avg', 'ep_fedproto'), ('fjord_federated_avg', 'fjord_federated')]:
    m = bundle.get(key, {})
    if isinstance(m, dict) and 'f1' in m:
        print(f'  {name}: F1={m["f1"]:.4f}')
print(f'  MAIN_SEEDS (headline, from NB2) = {MAIN_SEEDS}')
print(f'  fixed-tier F1 by dim: ' +
      ', '.join(f'd{d}={fixed_tier_avg[d]["f1"]:.4f}' for d in device_tier_dims_nb))
print(f'  scalability sweep client counts: {scalability_n_clients}')


## Diagnostic A — SSL Isolation

Validates that the pre-warm-only design is optimal. `Diag A1` = no SSL at
all. `Diag A2` = pre-warm only, which is provably identical to `CFG` itself
(asserted in-cell) — so it doubles as the `Full-Backbone (CFG)` reference row
used again in the Ablation Study below (aliased there, not re-run).

*(The old `Diag A3`, 'in-round SSL restored', is removed here — see the
notebook-level note at the top for why.)*

In [ ]:
# ── Diagnostic A: SSL Isolation ───────────────────────────────────────────────
# Validates that the pre-warm-only design is optimal.
# Diag A1 = no SSL at all (baseline).
# Diag A2 = pre-warm only (== CFG's own defaults -- see assertion below; this
#           IS the 'Full-Backbone (CFG)' reference row used again in the
#           Ablation Study further down, aliased there rather than re-run).
#
# EP-FedProto-v3 NB3 fix: the old 'Diag A3' (in-round SSL restored) referenced
# an ExperimentConfig(use_ssl_inround=True) flag. In-round SSL was REMOVED
# ENTIRELY from the codebase (see ExperimentConfig / ssl_pretrain_phase
# docstrings: "in-round SSL: REMOVED (pre-warm only)") -- there is no such
# field on ExperimentConfig any more, so constructing that config raises
# TypeError. Diag A3 tested a code path that no longer exists; removed rather
# than papered over with a fake flag. Diag A1 vs Diag A2 (== CFG) already
# answers the isolation question this diagnostic exists for: does pre-warm-only
# SSL help over no SSL at all.

print('=== Diagnostic A: SSL Isolation ===')
t_diag = time.time()

cfg_no_ssl       = ExperimentConfig(use_ssl=False)
cfg_prewarm_only = ExperimentConfig(use_ssl=True, ssl_pretrain_rounds=6, ssl_epochs=20)  # == CFG defaults

# Sanity-check cfg_prewarm_only really is identical to CFG (both are plain
# ExperimentConfig() with SSL fields set to their own defaults) before relying
# on that equivalence to skip a duplicate run in the Ablation Study below.
assert config_hash(cfg_prewarm_only) == config_hash(CFG), (
    "cfg_prewarm_only no longer matches CFG -- Diag A2 / 'Full-Backbone (CFG)' "
    "alias in the Ablation Study cell is unsafe; investigate before proceeding.")

diag_a1_runs, diag_a2_runs = [], []
for s in CFG.seeds:
    diag_a1_runs.append(
        run_or_load(f'diagv10_a1_s{s}',
                    lambda s=s: run_pgfcl(elliptic_data, elliptic_clients, DEVICE,
                                          cfg_no_ssl, seed=s, verbose=False,
                                          label='Diag-A1 No-SSL')[0]))
    # Diag A2 == CFG exactly (asserted above) -- cached under 'full_backbone_cfg_s{s}'
    # so the Ablation Study's 'Full-Backbone (CFG)' row can alias these same runs
    # instead of retraining an identical config.
    diag_a2_runs.append(
        run_or_load(f'full_backbone_cfg_s{s}',
                    lambda s=s: run_pgfcl(elliptic_data, elliptic_clients, DEVICE,
                                          cfg_prewarm_only, seed=s, verbose=False,
                                          label='Diag-A2 PrewarmOnly (== Full-Backbone CFG)')[0],
                    expected_hash=config_hash(CFG)))

diag_a1_avg = avg_metrics(diag_a1_runs)
diag_a2_avg = avg_metrics(diag_a2_runs)
print(f'  Diag A1 (No SSL)              F1={diag_a1_avg["f1"]:.4f}\u00b1{diag_a1_avg["f1_std"]:.4f}')
print(f'  Diag A2 (Pre-warm only == CFG) F1={diag_a2_avg["f1"]:.4f}\u00b1{diag_a2_avg["f1_std"]:.4f}')
print(f'  Diagnostic A done in {(time.time()-t_diag)/60:.1f} min')

# FIX A2: Wilcoxon non-parametric test (appropriate for n<10)
_a1_f1s = diag_a1_avg.get('f1_vals', [])
_a2_f1s = diag_a2_avg.get('f1_vals', [])
if len(_a1_f1s) >= 3 and len(_a2_f1s) >= 3:
    try:
        _n = min(len(_a2_f1s), len(_a1_f1s))
        _w, _p = scipy_stats.wilcoxon(_a2_f1s[:_n], _a1_f1s[:_n])
        print(f'  Wilcoxon Full-Backbone(CFG) vs No-SSL: stat={_w:.3f}  p={_p:.4f}')
    except Exception as _we:
        print(f'  Wilcoxon skipped: {_we}')

print()
print('Expected: Diag A2 (== CFG) is the reference; if Diag A1 (No-SSL) scores')
print('significantly higher, investigate pre-warm warmup length.')


## Ablation Study — Existing 9 Ablations (unchanged from v13) + Full-Backbone Reference

No-SSL, No-Proto, No-FedProx, No-ContribAgg, No-GradGuard, No-FocalLoss,
With-LabelProp, No-FedPer, No-EMA — same configs, same tiered-seed budget
(5 seeds for the two headline architectural claims No-Proto/No-FedPer, 3
seeds for the rest) as v13. See the in-cell note for the one change (the
`Full-Backbone (CFG)` reference row's source).

In [ ]:
# -- Ablation Study (existing 9 ablations, UNCHANGED from v13; see EP-FedProto- --
# -- specific ablations further below for the 3 new NB3 arms) ------------------
# v13 dedup (kept as-is): 'No-SSL' is an exact-duplicate config of a run we
# already have cached (Diag A1 above) -- aliased instead of re-run.
# v13 tiered seeds (kept as-is): the two configs tied to the backbone's
# headline architectural claims (No-Proto, No-FedPer) keep the full 5 seeds;
# the 6 secondary configs use 3 seeds each.
# FIX: config_hash is printed per variant so mismatched configs are detectable.
#
# EP-FedProto-v3 NB3 note (the only change from v13): 'Full PGFCL v10' used to
# alias `pgfcl_runs` loaded from the old PGFCL-lineage NB2 bundle. This
# notebook's NB2 bundle (`ep_fedproto_nb2_v2`) never ran that plain-CFG,
# non-device-tiered backbone config under that name -- its main new-work run
# is EP-FedProto (device-tiered, nested loss), a different config entirely.
# So there is nothing to alias here without inventing data that doesn't
# exist. Renamed to 'Full-Backbone (CFG)' and computed + cached directly
# (still literally `CFG`, still FULL_SEEDS, still run via the same
# `run_pgfcl` call as every other ablation row) -- same reference-row role
# as before (the "everything on, no device-tiering" backbone that the 9
# flag-ablations perturb one at a time from), just honestly sourced.

FULL_SEEDS     = CFG.seeds          # (42, 123, 7, 456, 789)
REDUCED_SEEDS  = CFG.seeds[:3]       # (42, 123, 7) -- first 3, same seeds used elsewhere

print(f'Full-rigor seeds (headline claims): {FULL_SEEDS}')
print(f'Reduced seeds (secondary ablations): {REDUCED_SEEDS}')
print('Running ablation...')
t2 = time.time()

ablation_configs = [
    ('No-SSL',         ExperimentConfig(use_ssl=False,  use_protos=True,  use_fedprox=True), FULL_SEEDS),
    ('No-Proto',       ExperimentConfig(use_ssl=True,   use_protos=False, use_fedprox=True), FULL_SEEDS),
    ('No-FedProx',     ExperimentConfig(use_ssl=True,   use_protos=True,  use_fedprox=False), REDUCED_SEEDS),
    ('No-ContribAgg',  ExperimentConfig(use_contrib_agg=False, use_grad_guard=False), REDUCED_SEEDS),
    ('No-GradGuard',   ExperimentConfig(use_grad_guard=False), REDUCED_SEEDS),
    ('No-FocalLoss',   ExperimentConfig(use_focal_loss=False), REDUCED_SEEDS),
    ('With-LabelProp', ExperimentConfig(use_label_prop=True), REDUCED_SEEDS),   # re-enable LP to measure its cost vs v10 default (LP off)
    ('No-FedPer',      ExperimentConfig(use_fedper=False), FULL_SEEDS),
    # FIX B3: EMA ablation -- ema_momentum=0.0 means raw round-mean (no smoothing)
    ('No-EMA',         ExperimentConfig(ema_momentum=0.0), REDUCED_SEEDS),
    ('Full-Backbone (CFG)', CFG, FULL_SEEDS),
]

# Sanity-check both aliases really are duplicates before trusting the shortcuts.
_no_ssl_cfg   = ExperimentConfig(use_ssl=False, use_protos=True, use_fedprox=True)
_no_ssl_hash  = config_hash(_no_ssl_cfg)
_diag_a1_hash = config_hash(cfg_no_ssl)
_full_hash    = config_hash(CFG)
_diag_a2_hash = config_hash(cfg_prewarm_only)
assert _no_ssl_hash == _diag_a1_hash, (
    f"'No-SSL' ablation config no longer matches Diag A1 config "
    f"({_no_ssl_hash} != {_diag_a1_hash}) -- alias is unsafe, remove it and let it run.")
assert _full_hash == _diag_a2_hash, (
    f"'Full-Backbone (CFG)' no longer matches Diag A2's config "
    f"({_full_hash} != {_diag_a2_hash}) -- alias is unsafe, remove it and let it run.")
print(f'  [alias check] No-SSL == Diag A1 (hash {_no_ssl_hash}): OK')
print(f'  [alias check] Full-Backbone (CFG) == Diag A2 (hash {_full_hash}): OK')

ablation_results = {}
for abl_label, abl_cfg, abl_seeds in ablation_configs:
    chash = config_hash(abl_cfg)
    print(f'\n-- Ablation: {abl_label} (hash={chash}, seeds={abl_seeds}) --')

    # v13 dedup (kept): reuse already-cached Diag-A1 runs for the identical No-SSL config.
    # NB3 addition: 'Full-Backbone (CFG)' reuses Diag A2 for the same reason
    # (identical config, asserted above) instead of retraining CFG a third
    # time (Diag A2 == CFG == this row).
    if abl_label == 'No-SSL':
        runs = diag_a1_runs
        print(f'  [alias] reusing diag_a1_runs ({len(runs)} seeds) -- identical config to Diag A1')
    elif abl_label == 'Full-Backbone (CFG)':
        runs = diag_a2_runs
        print(f'  [alias] reusing diag_a2_runs ({len(runs)} seeds) -- identical config to Diag A2 (CFG)')
    else:
        runs = []
        for s in abl_seeds:
            ckpt_key = f'ablepfp3_{abl_label.replace(" ","_")}_h{chash}_s{s}'
            result = run_or_load(
                ckpt_key,
                lambda s=s, c=abl_cfg, l=abl_label: run_pgfcl(
                    elliptic_data, elliptic_clients, DEVICE,
                    c, seed=s, verbose=False, label=l)
            )
            m = result[0] if isinstance(result, tuple) else result
            stored_hash = m.get('config_hash', 'unknown')
            if stored_hash != chash:
                print(f'  WARNING: stored hash {stored_hash} != expected {chash} -- cache mismatch, re-running')
                os.remove(os.path.join(LOCAL_CKPT_DIR, ckpt_key.replace(' ','_').replace('/','_')+'.pkl'))
                _rerun = run_pgfcl(elliptic_data, elliptic_clients, DEVICE,
                                   abl_cfg, seed=s, verbose=False, label=abl_label)
                save_ckpt(ckpt_key, _rerun)
                m = _rerun[0] if isinstance(_rerun, tuple) else _rerun
            runs.append(m)

    ablation_results[abl_label] = avg_metrics(runs)
    mv = ablation_results[abl_label]
    print(f'  F1={mv["f1"]:.4f}+/-{mv["f1_std"]:.4f} | AUC={mv["auc"]:.4f} | Prec={mv["prec"]:.4f} | Rec={mv["rec"]:.4f} | n_seeds={len(runs)}')

print(f'\nAblation done in {(time.time()-t2)/60:.1f} min')


## EP-FedProto-Specific Ablations — No-Nesting, Uniform-Agg, Full-EP-FedProto (NB3 new work)

Isolates EP-FedProto's two new mechanisms independently, holding everything
else (device tiers, backbone architecture, schedule) fixed at `CFG`:

- **No-Nesting** — single-budget loss at each client's own tier, rank-aware
  aggregator unchanged. Isolates the Matryoshka multi-budget loss's
  contribution.
- **Uniform-Agg** — nested multi-budget loss unchanged, plain size-weighted
  dimension-wise aggregation instead of rank-aware ContribAgg. Isolates the
  aggregator's contribution.
- **Full-EP-FedProto** — the reference row (both mechanisms on), aliased
  directly from NB2's headline run rather than re-run.

In [ ]:
# ── EP-FedProto-specific ablations (NB3 new work) ──────────────────────────────
# These isolate EP-FedProto's two new mechanisms independently:
#   - the multi-budget Matryoshka loss (vs a single-budget loss at each
#     client's own tier)      -> 'No-Nesting'
#   - the rank-aware, quality-weighted aggregator (vs plain size-weighted
#     dimension-wise aggregation) -> 'Uniform-Agg'
# 'Full-EP-FedProto' is the reference row (nested=True, agg_fn=rank_aware_contrib_agg
# -- exactly NB2's headline run) and is ALIASED from the NB2 bundle, not re-run.
#
# Seed budget: MAIN_SEEDS (matches NB2's headline EP-FedProto/FjORD-federated
# runs, so all three of this ablation's rows are directly comparable to each
# other AND to the headline numbers on identical seeds).

print(f'EP-FedProto ablation seeds: {MAIN_SEEDS}')
print('Running EP-FedProto-specific ablations (No-Nesting, Uniform-Agg)...')
t3 = time.time()

_ep_hash = config_hash(CFG)  # all three arms train under plain CFG (device-tiered)

# -- Full-EP-FedProto: alias, NOT re-run --------------------------------------
full_ep_fedproto_runs = ep_fedproto_runs
full_ep_fedproto_avg  = ep_fedproto_avg
print(f'  [alias] Full-EP-FedProto reuses ep_fedproto_runs from the NB2 bundle '
      f'({len(full_ep_fedproto_runs)} seeds)')

# -- No-Nesting: single-budget loss, rank-aware aggregator unchanged ----------
no_nesting_runs = []
for s in MAIN_SEEDS:
    result = run_or_load(
        f'no_nesting_s{s}',
        lambda s=s: run_ep_fedproto(
            elliptic_data, elliptic_clients, DEVICE, CFG, seed=s,
            nested=False, verbose=False, label=f'No-Nesting seed={s}'
        ),
        expected_hash=_ep_hash
    )
    no_nesting_runs.append(result[0])
no_nesting_avg = avg_metrics(no_nesting_runs)
print(f'  No-Nesting    F1={no_nesting_avg["f1"]:.4f}+/-{no_nesting_avg["f1_std"]:.4f} '
      f'| AUC={no_nesting_avg["auc"]:.4f} | n_seeds={len(no_nesting_runs)}')

# -- Uniform-Agg: nested loss unchanged, plain size-weighted dim-wise agg -----
uniform_agg_runs = []
for s in MAIN_SEEDS:
    result = run_or_load(
        f'uniform_agg_s{s}',
        lambda s=s: run_ep_fedproto(
            elliptic_data, elliptic_clients, DEVICE, CFG, seed=s,
            agg_fn=uniform_dimwise_agg, verbose=False, label=f'Uniform-Agg seed={s}'
        ),
        expected_hash=_ep_hash
    )
    uniform_agg_runs.append(result[0])
uniform_agg_avg = avg_metrics(uniform_agg_runs)
print(f'  Uniform-Agg   F1={uniform_agg_avg["f1"]:.4f}+/-{uniform_agg_avg["f1_std"]:.4f} '
      f'| AUC={uniform_agg_avg["auc"]:.4f} | n_seeds={len(uniform_agg_runs)}')

print(f'  Full-EP-FedProto F1={full_ep_fedproto_avg["f1"]:.4f}+/-{full_ep_fedproto_avg["f1_std"]:.4f} (reference row)')

epfp_ablation_results = {
    'Full-EP-FedProto': full_ep_fedproto_avg,
    'No-Nesting':       no_nesting_avg,
    'Uniform-Agg':      uniform_agg_avg,
}

print(f'\nEP-FedProto-specific ablations done in {(time.time()-t3)/60:.1f} min')
print('Interpretation: Full-EP-FedProto > No-Nesting isolates the nested-loss '
      "contribution; Full-EP-FedProto > Uniform-Agg isolates the rank-aware "
      "aggregator's contribution.")


## Mandatory Multi-Seed Core Comparison + Statistical Tests (NB3 new work)

NB2's headline EP-FedProto / FjORD-federated runs used 3 seeds — enough to
see a trend, not enough for a defensible paired significance test. This
section extends both to the same 5-seed budget FedProto already has (reusing
the 3 already-cached seeds, only training the 2 new ones), then reports
mean ± std, a paired t-test, a Wilcoxon signed-rank test, and Cohen's d
(paired) for EP-FedProto vs FedProto and EP-FedProto vs FjORD-federated.

In [ ]:
# ── Mandatory multi-seed core comparison (NB3 new work) ────────────────────────
# NB2's headline EP-FedProto / FjORD-federated runs used MAIN_SEEDS = CFG.seeds[:3]
# (3 seeds) -- adequate to establish the trend, but under-powered for the
# paired significance tests a paper needs. FedProto (from NB1, forwarded
# through the NB2 bundle) already has the full 5-seed budget. This cell
# extends EP-FedProto and FjORD-federated to the same FULL_SEEDS = CFG.seeds,
# reusing the 3 already-cached seeds via run_or_load (checkpoint keys match
# NB2's naming exactly) and only training the 2 new seeds.

print(f'Extending core-comparison seed budget: {MAIN_SEEDS} -> {FULL_SEEDS}')
t4 = time.time()

_ep_hash    = config_hash(CFG)
_fjord_hash = config_hash(CFG)

ep_fedproto_runs_full = []
for s in FULL_SEEDS:
    result = run_or_load(
        f'ep_fedproto_s{s}',
        lambda s=s: run_ep_fedproto(
            elliptic_data, elliptic_clients, DEVICE, CFG, seed=s,
            verbose=False, label=f'EP-FedProto seed={s}'
        ),
        expected_hash=_ep_hash
    )
    ep_fedproto_runs_full.append(result[0])

fjord_federated_runs_full = []
for s in FULL_SEEDS:
    result = run_or_load(
        f'fjord_federated_s{s}',
        lambda s=s: (run_fjord_gat(elliptic_data, elliptic_clients, DEVICE, CFG, s),),
        expected_hash=_fjord_hash
    )
    fjord_federated_runs_full.append(result[0])

ep_fedproto_avg_full    = avg_metrics(ep_fedproto_runs_full)
fjord_federated_avg_full = avg_metrics(fjord_federated_runs_full)

print(f'\nEP-FedProto   (n={len(ep_fedproto_runs_full)}) F1={ep_fedproto_avg_full["f1"]:.4f}+/-{ep_fedproto_avg_full["f1_std"]:.4f}')
print(f'FedProto      (n={len(fedproto_runs)}) F1={fedproto_avg["f1"]:.4f}+/-{fedproto_avg["f1_std"]:.4f}')
print(f'FjORD-federated (n={len(fjord_federated_runs_full)}) F1={fjord_federated_avg_full["f1"]:.4f}+/-{fjord_federated_avg_full["f1_std"]:.4f}')
print(f'(seed extension done in {(time.time()-t4)/60:.1f} min)')


def paired_stats(f1_a, f1_b, name_a, name_b):
    """Paired t-test, Wilcoxon signed-rank, and Cohen's d (paired) on matched
    per-seed F1 arrays. Requires f1_a/f1_b to be aligned to the SAME seeds
    in the SAME order (true here: both loops above iterate FULL_SEEDS in
    order, and fedproto_runs was built the same way over CFG.seeds in NB1)."""
    n = min(len(f1_a), len(f1_b))
    a, b = np.array(f1_a[:n]), np.array(f1_b[:n])
    diff = a - b
    out = {'n': n, 'mean_diff': float(diff.mean()), 'std_diff': float(diff.std(ddof=1)) if n > 1 else 0.0}
    if n >= 2:
        try:
            t_stat, t_p = scipy_stats.ttest_rel(a, b)
            out['t_stat'], out['t_p'] = float(t_stat), float(t_p)
        except Exception as e:
            out['t_stat'], out['t_p'] = None, None
            print(f'  [t-test skipped: {e}]')
    if n >= 3:
        try:
            w_stat, w_p = scipy_stats.wilcoxon(a, b)
            out['wilcoxon_stat'], out['wilcoxon_p'] = float(w_stat), float(w_p)
        except Exception as e:
            out['wilcoxon_stat'], out['wilcoxon_p'] = None, None
            print(f'  [Wilcoxon skipped: {e}]')
    # Cohen's d for paired samples: mean(diff) / std(diff)
    out['cohens_d'] = float(diff.mean() / diff.std(ddof=1)) if n > 1 and diff.std(ddof=1) > 1e-12 else None

    print(f'\n-- {name_a} vs {name_b} (n={n} paired seeds) --')
    print(f'  mean F1 diff = {out["mean_diff"]:+.4f} (std {out["std_diff"]:.4f})')
    if out.get('t_p') is not None:
        print(f'  paired t-test:   t={out["t_stat"]:.3f}  p={out["t_p"]:.4f}')
    if out.get('wilcoxon_p') is not None:
        print(f'  Wilcoxon:        stat={out["wilcoxon_stat"]:.3f}  p={out["wilcoxon_p"]:.4f}')
    if out.get('cohens_d') is not None:
        print(f'  Cohen\'s d (paired) = {out["cohens_d"]:.3f}')
    return out


ep_f1  = [r['f1'] for r in ep_fedproto_runs_full]
fp_f1  = [r['f1'] for r in fedproto_runs]
fj_f1  = [r['f1'] for r in fjord_federated_runs_full]

core_comparison_stats = {
    'ep_vs_fedproto': paired_stats(ep_f1, fp_f1, 'EP-FedProto', 'FedProto'),
    'ep_vs_fjord':    paired_stats(ep_f1, fj_f1, 'EP-FedProto', 'FjORD-federated'),
}


## Non-IID Severity Sweep (NB3 new work)

The existing temporal split is a single fixed point of non-IID-ness (pure
chronological order — the CAVEAT already documented at the split cell above).
This section introduces `temporal_federated_split_severity`, a parameterized
family of splits that interpolates between that pure-chronological ordering
(`severity=1.0`, verified identical to the original split) and an
approximately-IID random ordering (`severity=0.0`), and sweeps the
EP-FedProto vs FedProto F1 gap across severities.

In [ ]:
# ── Non-IID severity sweep: split function (NB3 new work) ──────────────────────
# temporal_federated_split (above) is a fixed point: pure chronological order,
# maximal client heterogeneity (the CAVEAT already documented there). To sweep
# non-IID *severity* we need a family of splits parameterized by how close to
# that chronological ordering vs. a random (approximately client-IID) ordering
# each client's data comes from.
#
# temporal_federated_split_severity(severity=1.0) is IDENTICAL to
# temporal_federated_split (same sort key, same stratified train/test carve-up
# per client) -- verified below, not just claimed. severity=0.0 replaces the
# chronological sort key with a random permutation, so clients get an
# approximately-uniform mix of time windows instead of contiguous blocks.
# Intermediate severities linearly blend the two sort keys.

def temporal_federated_split_severity(data: Data, cfg: ExperimentConfig,
                                       severity: float = 1.0, seed: int = 0,
                                       verbose: bool = True):
    labels    = data.y.numpy()
    timesteps = data.timestep.numpy()
    valid_idx = np.where(labels >= 0)[0]

    if severity >= 1.0:
        sorted_idx = valid_idx[np.argsort(timesteps[valid_idx])]
    else:
        rng = np.random.RandomState(seed)
        # Rank-normalize both orderings to [0, 1] so they're on the same
        # scale before blending -- timesteps aren't uniformly spaced, ranks are.
        temporal_rank = np.argsort(np.argsort(timesteps[valid_idx])).astype(np.float64)
        random_rank   = rng.permutation(len(valid_idx)).astype(np.float64)
        denom = max(len(valid_idx) - 1, 1)
        temporal_rank /= denom
        random_rank   /= denom
        combined_key = severity * temporal_rank + (1.0 - severity) * random_rank
        sorted_idx = valid_idx[np.argsort(combined_key)]

    splits = np.array_split(sorted_idx, cfg.n_clients)

    def strat_split(arr):
        if len(arr) == 0:
            return arr, arr
        n_te = max(1, int(len(arr) * cfg.test_ratio))
        return arr[:-n_te], arr[-n_te:]

    clients = []
    for i, split in enumerate(splits):
        illicit        = split[labels[split] == 1]
        licit          = split[labels[split] == 0]
        ill_tr, ill_te = strat_split(illicit)
        lic_tr, lic_te = strat_split(licit)
        tr = np.concatenate([ill_tr, lic_tr])
        te = np.concatenate([ill_te, lic_te])
        clients.append({
            'id':         i,
            'train_mask': make_mask(data.num_nodes, tr),
            'test_mask':  make_mask(data.num_nodes, te),
            'n_train':    len(tr),
            'n_test':     len(te),
        })
        if verbose:
            print(f'  Client {i}: train={len(tr):5d} test={len(te):4d} '
                  f'ill_train={len(ill_tr):4d} ill_test={len(ill_te):3d}')
    return clients


# Verify severity=1.0 reproduces temporal_federated_split's client contents
# EXACTLY (not just "similar non-IID-ness") before trusting the sweep below.
_ref_clients  = elliptic_clients
_sev1_clients = temporal_federated_split_severity(elliptic_data, CFG, severity=1.0, verbose=False)
for _rc, _sc in zip(_ref_clients, _sev1_clients):
    assert torch.equal(_rc['train_mask'], _sc['train_mask']), \
        'severity=1.0 split diverges from temporal_federated_split -- fix before sweeping.'
    assert torch.equal(_rc['test_mask'], _sc['test_mask']), \
        'severity=1.0 split diverges from temporal_federated_split -- fix before sweeping.'
print('[check] temporal_federated_split_severity(severity=1.0) == temporal_federated_split: OK')


In [ ]:
# ── Non-IID severity sweep: runs (NB3 new work) ─────────────────────────────────
# Compute-scoping decision (stated openly, same pattern as SWEEP_CFG/EDGE_CFG
# elsewhere in this project): severity is swept at 2 seeds per point rather
# than the full 5-seed budget -- this sweep is about the SHAPE of the
# F1-vs-severity curve for the core comparison (EP-FedProto vs FedProto),
# not a headline number, so it doesn't need the same statistical power as
# the paired core-comparison tests above. Uses CFG's normal architecture and
# schedule (not SWEEP_CFG) since severity only changes the DATA split, not
# the model.

NONIID_SEVERITIES = (1.0, 0.7, 0.4, 0.1)  # 1.0 = original temporal (max non-IID), 0.1 = near-IID
NONIID_SEEDS       = CFG.seeds[:2]

print(f'Non-IID severity sweep: severities={NONIID_SEVERITIES}, seeds={NONIID_SEEDS}')
t5 = time.time()

noniid_splits = {}
for sev in NONIID_SEVERITIES:
    print(f'\n=== severity={sev} ===')
    noniid_splits[sev] = temporal_federated_split_severity(
        elliptic_data, CFG, severity=sev, seed=0, verbose=True)

noniid_results = {}  # {severity: {'ep_fedproto': avg_metrics, 'fedproto': avg_metrics}}
_fedproto_cfg = ExperimentConfig(use_ssl=False, use_protos=True, use_fedprox=False)
_fedproto_hash = config_hash(_fedproto_cfg)
_ep_hash = config_hash(CFG)

for sev in NONIID_SEVERITIES:
    sev_clients = noniid_splits[sev]
    ep_runs, fp_runs = [], []
    for s in NONIID_SEEDS:
        ep_result = run_or_load(
            f'noniid_sev{sev}_ep_fedproto_s{s}',
            lambda s=s, sc=sev_clients: run_ep_fedproto(
                elliptic_data, sc, DEVICE, CFG, seed=s,
                verbose=False, label=f'EP-FedProto sev={sev} seed={s}'
            ),
            expected_hash=_ep_hash
        )
        ep_runs.append(ep_result[0])

        fp_result = run_or_load(
            f'noniid_sev{sev}_fedproto_s{s}',
            lambda s=s, sc=sev_clients: run_pgfcl(
                elliptic_data, sc, DEVICE, _fedproto_cfg, seed=s,
                verbose=False, label=f'FedProto sev={sev} seed={s}')[0],
            expected_hash=_fedproto_hash
        )
        fp_runs.append(fp_result)

    noniid_results[sev] = {
        'ep_fedproto': avg_metrics(ep_runs),
        'fedproto':    avg_metrics(fp_runs),
    }
    ep_f1 = noniid_results[sev]['ep_fedproto']['f1']
    fp_f1 = noniid_results[sev]['fedproto']['f1']
    print(f'  severity={sev}: EP-FedProto F1={ep_f1:.4f} | FedProto F1={fp_f1:.4f} | gap={ep_f1-fp_f1:+.4f}')

print(f'\nNon-IID severity sweep done in {(time.time()-t5)/60:.1f} min')
print('Interpretation: does the EP-FedProto-vs-FedProto F1 gap hold, shrink, or '
      'grow as clients become more IID (severity -> 0)?')


## Skewed Device-Tier Distribution — 60/30/10 (NB3 new work)

Realistic device-fleet assumption, stated openly: most real client fleets
skew toward constrained hardware rather than a uniform split across
`DEVICE_TIER_DIMS`. Models a 60% / 30% / 10% / 0% skew toward the low end
(nobody in this fleet has full-width capacity) and re-runs EP-FedProto under
it — also sets up the Failure-Case Analysis below, which is built on exactly
this kind of thin, skewed fleet.

In [ ]:
# ── Skewed device-tier distribution (NB3 new work) ──────────────────────────────
# Realistic device-fleet assumption, stated openly: most real client fleets
# skew toward constrained hardware, not a uniform 25/25/25/25 split across
# DEVICE_TIER_DIMS=(8,16,32,64). We model a 60/30/10 skew toward the low end
# -- 60% of clients at the smallest tier (d=8), 30% at d=16, 10% at d=32, and
# (since 60/30/10 only specifies three numbers) 0% at the largest tier d=64,
# i.e. nobody in this fleet has enough capacity for the full embedding width.
# This is also the setup the Failure-Case Analysis below is built on, since
# it's exactly the kind of skew that can leave a nested aggregation segment
# with very few (or, as shown below, exactly one) contributing clients.

SKEWED_TIER_DISTRIBUTION = (0.6, 0.3, 0.1, 0.0)  # matches DEVICE_TIER_DIMS order (8,16,32,64)
assert len(SKEWED_TIER_DISTRIBUTION) == len(DEVICE_TIER_DIMS)
assert abs(sum(SKEWED_TIER_DISTRIBUTION) - 1.0) < 1e-9

_skew_preview = assign_device_tiers(elliptic_clients, DEVICE_TIER_DIMS,
                                     distribution=SKEWED_TIER_DISTRIBUTION, seed=CFG.seeds[0])
print(f'Skewed device-tier distribution {SKEWED_TIER_DISTRIBUTION} over DEVICE_TIER_DIMS={DEVICE_TIER_DIMS}')
print(f'Preview @ seed={CFG.seeds[0]}, n_clients={CFG.n_clients}: {_skew_preview}')

SKEWED_SEEDS = CFG.seeds[:2]  # same compute-scoping rationale as the non-IID sweep above
_ep_hash = config_hash(CFG)

print(f'\nRunning EP-FedProto under the skewed device-tier distribution (seeds={SKEWED_SEEDS})...')
t6 = time.time()

skewed_tier_runs = []
for s in SKEWED_SEEDS:
    result = run_or_load(
        f'skewed_tier_ep_fedproto_s{s}',
        lambda s=s: run_ep_fedproto(
            elliptic_data, elliptic_clients, DEVICE, CFG, seed=s,
            tier_distribution=SKEWED_TIER_DISTRIBUTION,
            verbose=False, label=f'EP-FedProto skewed-tier seed={s}'
        ),
        expected_hash=_ep_hash
    )
    skewed_tier_runs.append(result)

skewed_tier_metrics = [r[0] for r in skewed_tier_runs]
skewed_tier_avg = avg_metrics(skewed_tier_metrics)
print(f'\nSkewed-tier EP-FedProto: F1={skewed_tier_avg["f1"]:.4f}+/-{skewed_tier_avg["f1_std"]:.4f} '
      f'(uniform-tier Full-EP-FedProto reference: F1={full_ep_fedproto_avg["f1"]:.4f})')
print(f'(done in {(time.time()-t6)/60:.1f} min)')

for r in skewed_tier_metrics:
    print(f"  seed device_tier_map: {r.get('device_tier_map')}")


## Failure-Case Analysis — Sparse High-Dim ContribAgg (NB3 new work)

`rank_aware_contrib_agg` renormalizes its ContribAgg quality-weighted score
over only the clients *eligible* for a given dimension segment. When a
segment has exactly one eligible client, that renormalization is a ratio of a
value to itself: the resulting weight collapses to ≈1.0 for *any* positive
quality score, regardless of how anomalous that client's update actually is —
ContribAgg's entire purpose (down-weighting a low-quality/outlier
contribution) is structurally unavailable in a singleton segment.

Part 1 demonstrates this directly on `rank_aware_contrib_agg` with a
synthetic corrupted update, contrasted against the same corruption in a
healthy multi-client segment. Part 2 checks whether this actually occurs
under the skewed 60/30/10/0 fleet from the previous cell, at this project's
default `n_clients=4`.

In [ ]:
# ── Failure-case analysis: sparse high-dim ContribAgg (NB3 new work) ───────────
# rank_aware_contrib_agg (NB2) computes each segment's aggregation weight as
#     weight_i = (size_i**0.5 * quality_i) / sum_j(size_j**0.5 * quality_j)
# over only the clients eligible for that segment. When a segment has exactly
# ONE eligible client, weight_i is a ratio of a value to itself: it collapses
# to (approximately) 1.0 for ANY quality_i > 0 -- the renormalization erases
# whatever the cosine-similarity quality score computed. ContribAgg's entire
# purpose (down-weighting a low-quality/outlier client) is structurally
# unavailable in a singleton segment. This is a real property of the
# aggregator's math, not a contrived edge case -- part (2) below shows it
# actually occurs under the skewed 60/30/10/0 distribution from the previous
# cell at the project's default n_clients=4.

# ── Part 1: mathematical demonstration on a synthetic scenario ────────────────
print('=== Part 1: synthetic singleton-segment vs multi-client-segment ===')

def _rand_proto(dim, device, gen, scale=1.0):
    return scale * torch.randn(dim, generator=gen, device=device)

_gen = torch.Generator(device='cpu').manual_seed(0)
_emb_dim = CFG.emb_dim  # 64
_device_cpu = torch.device('cpu')

# Scenario A: n=4 clients, tiers=[8,16,32,64] (one client per tier, matching
# the DEFAULT uniform round-robin assign_device_tiers behaviour) -- segment
# [32,64) has exactly ONE eligible client (the sole tier-64 client): singleton.
tiers_A = [8, 16, 32, 64]
sizes_A = [500, 500, 500, 500]
protos_A = []
for _ in tiers_A:
    protos_A.append({0: _rand_proto(_emb_dim, _device_cpu, _gen),
                      1: _rand_proto(_emb_dim, _device_cpu, _gen)})
prev_global_A = {0: _rand_proto(_emb_dim, _device_cpu, _gen),
                  1: _rand_proto(_emb_dim, _device_cpu, _gen)}

# Corrupt the tier-64 client's top segment [32:64) with a large outlier shift --
# stand-in for a diverged/poisoned/anomalous client update.
_corrupt_idx_A = tiers_A.index(64)
protos_A[_corrupt_idx_A][1] = protos_A[_corrupt_idx_A][1].clone()
protos_A[_corrupt_idx_A][1][32:64] += 50.0  # large corruption, should be flagged

agg_A = rank_aware_contrib_agg(protos_A, tiers_A, sizes_A, prev_global_A, CFG, tier_dims=DEVICE_TIER_DIMS)

# Recompute the effective per-client weight ContribAgg assigned to the
# corrupted client's top segment directly, to report the number (not just
# infer it from the aggregate).
def _segment_weight_for_client(client_idx, seg_lo, seg_hi, client_full_protos,
                                client_tier_dims, client_sizes, prev_global_full_protos, cfg):
    eligible = [i for i in range(len(client_full_protos)) if client_tier_dims[i] >= seg_hi]
    if not eligible:
        max_dim = max(client_tier_dims)
        eligible = [i for i in range(len(client_full_protos)) if client_tier_dims[i] == max_dim]
    weights = []
    for i in eligible:
        sz = client_sizes[i]
        if prev_global_full_protos is None:
            weights.append(sz ** 0.5)
            continue
        local_seg  = client_full_protos[i][1][seg_lo:seg_hi]
        global_seg = prev_global_full_protos[1][seg_lo:seg_hi]
        if local_seg.norm() < 1e-8 or global_seg.norm() < 1e-8:
            quality = 1.0
        else:
            cos_sim = F.cosine_similarity(local_seg.unsqueeze(0), global_seg.unsqueeze(0)).item()
            quality = cfg.contrib_floor + (1.0 - cfg.contrib_floor) * (cos_sim + 1.0) / 2.0
        weights.append((sz ** 0.5) * quality)
    total_w = sum(weights) + 1e-8
    if client_idx not in eligible:
        return None, len(eligible)
    return weights[eligible.index(client_idx)] / total_w, len(eligible)

w_singleton, n_eligible_singleton = _segment_weight_for_client(
    _corrupt_idx_A, 32, 64, protos_A, tiers_A, sizes_A, prev_global_A, CFG)
print(f'  Scenario A (singleton top segment, n_eligible={n_eligible_singleton}): '
      f'corrupted client\'s effective weight in [32:64) = {w_singleton:.4f}')
print(f'  -> ContribAgg could NOT meaningfully down-weight this corrupted update: '
      f'renormalizing a single eligible client always yields weight ~1.0.')

# Scenario B: same corruption, but n=8 clients with 2 clients at tier=64, so
# segment [32,64) has TWO eligible clients and a real consensus to compare
# against -- ContribAgg's cosine-similarity quality term can actually act.
tiers_B = [8, 8, 16, 16, 32, 32, 64, 64]
sizes_B = [500] * 8
protos_B = []
for _ in tiers_B:
    protos_B.append({0: _rand_proto(_emb_dim, _device_cpu, _gen),
                      1: _rand_proto(_emb_dim, _device_cpu, _gen)})
prev_global_B = {0: _rand_proto(_emb_dim, _device_cpu, _gen),
                  1: _rand_proto(_emb_dim, _device_cpu, _gen)}
_tier64_idxs_B = [i for i, t in enumerate(tiers_B) if t == 64]
_corrupt_idx_B = _tier64_idxs_B[0]
protos_B[_corrupt_idx_B][1] = protos_B[_corrupt_idx_B][1].clone()
protos_B[_corrupt_idx_B][1][32:64] += 50.0

w_healthy, n_eligible_healthy = _segment_weight_for_client(
    _corrupt_idx_B, 32, 64, protos_B, tiers_B, sizes_B, prev_global_B, CFG)
print(f'\n  Scenario B (multi-client top segment, n_eligible={n_eligible_healthy}): '
      f'corrupted client\'s effective weight in [32:64) = {w_healthy:.4f}')
print(f'  -> With a real consensus to compare against, the same corruption gets '
      f'down-weighted toward contrib_floor={CFG.contrib_floor} instead of ~1.0.')

singleton_beats_downweight = w_singleton > w_healthy + 0.05
print(f'\n  [check] singleton segment leaves corruption meaningfully less '
      f'down-weighted than the multi-client segment: {singleton_beats_downweight}')


In [ ]:
# ── Part 2: does this actually occur under the skewed device-tier fleet? ──────
print('=== Part 2: singleton-segment frequency under the skewed 60/30/10/0 fleet ===')

def segment_eligibility_counts(client_tier_dims, tier_dims=DEVICE_TIER_DIMS):
    """Mirrors rank_aware_contrib_agg's eligibility rule (including the
    fallback-to-max-tier branch) and returns {segment: n_eligible_clients}."""
    boundaries = [0] + list(tier_dims)
    counts = {}
    for seg_lo, seg_hi in zip(boundaries[:-1], boundaries[1:]):
        eligible = [i for i in range(len(client_tier_dims)) if client_tier_dims[i] >= seg_hi]
        if not eligible:
            max_dim = max(client_tier_dims)
            eligible = [i for i in range(len(client_tier_dims)) if client_tier_dims[i] == max_dim]
        counts[(seg_lo, seg_hi)] = len(eligible)
    return counts


any_singleton_observed = False
for r in skewed_tier_runs:
    m = r[0]
    tier_map = m.get('device_tier_map', {})
    tier_list = [tier_map[c['id']] for c in elliptic_clients]
    seg_counts = segment_eligibility_counts(tier_list, DEVICE_TIER_DIMS)
    print(f'  device tiers: {tier_list}  ->  segment eligibility: {seg_counts}')
    if any(c <= 1 for c in seg_counts.values()):
        any_singleton_observed = True

print(f'\n[finding] Under SKEWED_TIER_DISTRIBUTION={SKEWED_TIER_DISTRIBUTION} at the default '
      f'n_clients={CFG.n_clients}, at least one segment has <=1 eligible client in every '
      f'observed run: {any_singleton_observed}. With no client at the top tier (d=64) and '
      f'only ~1 client rounding into d=32 at this small n_clients, the top one or two '
      f'segments regularly fall into the failure mode demonstrated in Part 1.')

failure_case_summary = {
    'singleton_weight_demo':    float(w_singleton),
    'multi_client_weight_demo': float(w_healthy),
    'skewed_fleet_has_singleton_segment': bool(any_singleton_observed),
    'skewed_fleet_segment_eligibility': [
        segment_eligibility_counts(
            [r[0]['device_tier_map'][c['id']] for c in elliptic_clients], DEVICE_TIER_DIMS)
        for r in skewed_tier_runs
    ],
}
print('\nRecommendation for the writeup: report this as a known limitation of '
      'rank-aware ContribAgg under thin/skewed device fleets, e.g. a minimum-'
      'eligible-clients floor (fall back to Uniform-Agg-style size weighting, '
      'or borrow the segment below) rather than trusting a lone contributor.')


## Save NB3 Bundle

In [ ]:
nb3_bundle = {
    # ── carried over from NB2 (unchanged) ──
    'central_avg': central_avg, 'central_runs': central_runs,
    'localonly_avg': localonly_avg, 'localonly_runs': localonly_runs,
    'fedavg_avg': fedavg_avg, 'fedavg_runs': fedavg_runs,
    'fedsage_avg': fedsage_avg, 'fedsage_runs': fedsage_runs,
    'fedsage_gat_avg': fedsage_gat_avg, 'fedsage_gat_runs': fedsage_gat_runs,
    'fedproto_avg': fedproto_avg, 'fedproto_runs': fedproto_runs,
    'moon_avg': moon_avg, 'moon_runs': moon_runs,
    'scaffold_avg': scaffold_avg, 'scaffold_runs': scaffold_runs,
    'fixed_tier_avg': fixed_tier_avg, 'fixed_tier_runs': fixed_tier_runs,
    'edge_confirmation_run': edge_confirmation_run,
    'device_tier_dims': device_tier_dims_nb,
    'edge_tier_dims': edge_tier_dims,
    'scalability_n_clients': scalability_n_clients,
    'scalability_results': scalability_results,
    'ep_fedproto_avg': ep_fedproto_avg, 'ep_fedproto_runs': ep_fedproto_runs,
    'ep_fedproto_drift_all': ep_fedproto_drift_all,
    'ep_fedproto_f1c_all': ep_fedproto_f1c_all,
    'ep_fedproto_comm_all': ep_fedproto_comm_all,
    'ep_fedproto_saliency_all': ep_fedproto_saliency_all,
    'fjord_federated_avg': fjord_federated_avg, 'fjord_federated_runs': fjord_federated_runs,
    'edge_full_avg': edge_full_avg, 'edge_full_runs': edge_full_runs,
    'edge_cfg_hash': edge_cfg_hash,
    'main_seeds': MAIN_SEEDS,

    # ── NB3 new work: Diagnostic A (SSL isolation; A3 removed, see cell note) ──
    'diag_a1_avg': diag_a1_avg, 'diag_a1_runs': diag_a1_runs,
    'diag_a2_avg': diag_a2_avg, 'diag_a2_runs': diag_a2_runs,

    # ── NB3 new work: existing 9 ablations + Full-Backbone(CFG) reference row ──
    'ablation_results': ablation_results,
    'full_seeds': FULL_SEEDS,
    'reduced_seeds': REDUCED_SEEDS,

    # ── NB3 new work: 3 new EP-FedProto-specific ablation arms ──
    'epfp_ablation_results': epfp_ablation_results,
    'no_nesting_runs': no_nesting_runs,
    'uniform_agg_runs': uniform_agg_runs,
    'full_ep_fedproto_runs': full_ep_fedproto_runs,

    # ── NB3 new work: mandatory multi-seed core comparison + stats ──
    'ep_fedproto_runs_full': ep_fedproto_runs_full,
    'ep_fedproto_avg_full': ep_fedproto_avg_full,
    'fjord_federated_runs_full': fjord_federated_runs_full,
    'fjord_federated_avg_full': fjord_federated_avg_full,
    'core_comparison_stats': core_comparison_stats,

    # ── NB3 new work: non-IID severity sweep ──
    'noniid_severities': NONIID_SEVERITIES,
    'noniid_seeds': NONIID_SEEDS,
    'noniid_results': noniid_results,

    # ── NB3 new work: skewed device-tier distribution (60/30/10) ──
    'skewed_tier_distribution': SKEWED_TIER_DISTRIBUTION,
    'skewed_seeds': SKEWED_SEEDS,
    'skewed_tier_runs': skewed_tier_runs,
    'skewed_tier_avg': skewed_tier_avg,

    # ── NB3 new work: failure-case analysis (sparse high-dim ContribAgg) ──
    'failure_case_summary': failure_case_summary,
}
save_ckpt('nb3_bundle', nb3_bundle)
print('nb3_bundle saved.')
print('Next: Save & Run All -> add this notebook output as input to NB4 '
      '(headline plots, compute/communication tables, Pareto figure, '
      'scalability plot, non-IID + failure-case figures, edge-device results).')
